# Mean and variance in one pass: Welford meets the tree reduction

The textbook formula $\operatorname{Var}(x) = \mathbb{E}[x^2] - \mathbb{E}[x]^2$ needs one pass, but in `float32` it is useless when the mean is large compared with the spread: for data around $10^4$ with unit variance, $\mathbb{E}[x^2] \approx 10^8$ and `float32` keeps only about 7 digits, so the difference is noise.

The stable alternative tracks a triple $(c, \mu, M_2)$ (count, mean, sum of squared deviations from the mean). Two triples merge exactly:

$$c = c_a + c_b,\qquad \delta = \mu_b - \mu_a,\qquad \mu = \mu_a + \delta\,\frac{c_b}{c},\qquad M_2 = M_{2,a} + M_{2,b} + \delta^2\,\frac{c_a c_b}{c}$$

Adding a single value $x$ is the special case $(1, x, 0)$ (Welford's update). Because the merge is associative, it plugs straight into a GPU tree reduction.

## Task
Write `mean_var(x)` that returns `(mean, variance)` with the **population** variance $M_2 / c$ (NumPy's default, `ddof=0`).

1. **Stage 1**, `BLOCKS = 4` blocks of `TPB = 64` threads: each thread folds its grid-stride elements into a triple, the block tree-reduces 64 triples with the merge rule (three `float32` shared arrays), and thread 0 writes the block's triple to three partial arrays of length 4.
2. **Stage 2**, one block of 64 threads: merge the 4 partials the same way and write the final mean and $M_2/c$.

Threads with no elements hold $(0, 0, 0)$. The merge must handle $c = 0$ (two empty triples) without dividing by zero.

## Inputs and output
- `x`: `float32`, shape `(n,)`, `1 <= n <= 4096`. All arithmetic in `float32`; counts can be held as `float32` (exact up to $2^{24}$).
- Return a tuple of two Python floats. Compared with relative tolerance $10^{-4}$, absolute $10^{-5}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

BLOCKS, TPB = 4, 64

def mean_var(x):
    """x: float32 (n,), n >= 1. Returns (mean, population variance) as Python floats."""
    raise NotImplementedError

@cuda.jit(device=True)
def merge(ca, ma, qa, cb, mb, qb):
    """Merge triples (count, mean, M2). Returns the merged (count, mean, M2)."""
    return ca + cb, ma, qa   # TODO

@cuda.jit
def stage1(x, pc, pm, pq):
    pass

@cuda.jit
def stage2(pc, pm, pq, out):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0, 4.0], "dtype": "float32"}], "e": {"__tup__": [2.5, 1.25]}}, {"a": [{"__nd__": [5.0], "dtype": "float32"}], "e": {"__tup__": [5.0, 0.0]}}, {"a": [{"__nd__": [7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25, 7.25], "dtype": "float32"}], "e": {"__tup__": [7.25, 0.0]}}, {"a": [{"__nd__": [-1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0], "dtype": "float32"}], "e": {"__tup__": [0.0, 1.0]}}, {"a": [{"__nd__": [10000.244140625, 10000.677734375, 9999.4140625, 9999.091796875, 9998.0078125, 10000.9716796875, 10000.0166015625, 10000.2060546875, 9999.216796875, 10001.2265625, 10000.943359375, 9999.8779296875, 9999.4443359375, 9999.6435546875, 9999.201171875, 10000.0263671875, 9999.3837890625, 10000.5859375, 9999.640625, 9999.646484375, 10001.537109375, 9999.2744140625, 9998.431640625, 10000.3662109375, 9997.921875, 10000.2646484375, 10000.9814453125, 10000.0703125, 10000.0087890625, 9997.94921875, 10000.5556640625, 10000.396484375, 9998.828125, 10001.9130859375, 9998.7763671875, 9998.3203125, 9998.1884765625, 10000.0263671875, 10000.337890625, 10000.44140625, 9999.6630859375, 9999.1875, 10000.8427734375, 9999.1025390625, 9998.7900390625, 10001.4013671875, 10000.177734375, 10000.4716796875, 10000.6259765625, 9999.427734375, 9999.466796875, 10000.640625, 9999.3828125, 10000.3359375, 9999.91015625, 9998.8056640625, 10000.966796875, 9998.7109375, 10000.2333984375, 10000.0673828125, 10001.2666015625, 9998.880859375, 9999.45703125, 9999.8994140625, 10000.1806640625, 10000.8681640625, 10000.3115234375, 10000.0244140625, 10001.0283203125, 10002.095703125, 9998.64453125, 9999.994140625, 10000.759765625, 10000.0859375, 10001.4345703125, 9999.666015625, 9998.986328125, 10000.8701171875, 9997.2275390625, 9999.5244140625, 10002.3662109375, 10000.0263671875, 10000.357421875, 9999.048828125, 9999.9765625, 9999.857421875, 10001.8583984375, 9998.990234375, 9997.23828125, 10000.40625, 10001.5498046875, 9999.3115234375, 9998.1767578125, 9998.9267578125, 10000.43359375, 9999.4375, 10000.0419921875, 9997.642578125, 9999.4794921875, 10000.7880859375, 9999.7353515625, 9998.9755859375, 10001.0517578125, 10000.9873046875, 9999.5341796875, 9999.2578125, 10001.2783203125, 9999.4501953125, 9999.8369140625, 10001.80078125, 10000.1865234375, 9999.5048828125, 9998.0234375, 10000.03515625, 9999.9130859375, 9999.3759765625, 10000.92578125, 9998.708984375, 9998.4169921875, 9999.2119140625, 9999.9453125, 9998.4697265625, 9998.791015625, 9999.1220703125, 9999.9091796875, 9999.2294921875, 10000.90625, 9997.7119140625, 10000.3203125, 9999.3173828125, 10000.373046875, 10001.8837890625, 9999.8369140625, 10000.103515625, 10000.7421875, 9998.5791015625, 10000.568359375, 10000.72265625, 9999.771484375, 10000.5615234375, 10000.037109375, 9999.1455078125, 9998.59375, 10000.8896484375, 10001.689453125, 9999.123046875, 10000.2783203125, 9998.697265625, 9999.44140625, 9999.416015625, 9998.7734375, 9999.8388671875, 10000.90625, 10000.4755859375, 9999.3994140625, 10001.33984375, 10000.1376953125, 10001.7216796875, 9998.6142578125, 10002.939453125, 9999.74609375, 9998.7412109375, 9998.857421875, 10000.857421875, 10000.8310546875, 10000.73828125, 10000.53515625, 9998.7998046875, 9999.783203125, 9997.615234375, 9999.0546875, 10000.6005859375, 9998.052734375, 10002.30859375, 10001.201171875, 9998.791015625, 10000.6630859375, 10000.4951171875, 9999.8017578125, 10000.53125, 9999.67578125, 9999.9248046875, 10000.4951171875, 10000.4296875, 10000.291015625, 10000.3125, 10000.9384765625, 10000.7626953125, 9998.7939453125, 9999.2255859375, 10001.4482421875, 10001.55078125, 10000.5478515625, 9999.7177734375, 10000.8154296875, 10000.677734375, 9998.4501953125, 10000.06640625, 9998.5986328125, 9999.5390625, 10000.03125, 10001.720703125, 9997.8837890625, 9999.6435546875, 10000.6826171875, 10000.6669921875, 10000.8505859375, 9999.005859375, 9999.87109375, 9999.537109375, 10001.310546875, 9999.8623046875, 10000.017578125, 10000.19140625, 9998.6904296875, 10000.517578125, 9998.89453125, 10000.18359375, 9999.2744140625, 10001.1982421875, 10000.0380859375, 10001.9306640625, 9997.765625, 10001.3720703125, 10000.8603515625, 10000.29296875, 10002.20703125, 9998.572265625, 10000.212890625, 10001.2470703125, 10000.607421875, 9999.62109375, 10000.6484375, 9999.5849609375, 9999.6591796875, 9999.337890625, 10001.0546875, 10000.3125, 10002.5166015625, 10000.8466796875, 9998.5166015625, 10000.802734375, 9998.9111328125, 9999.7236328125, 10000.1416015625, 9998.9091796875, 9999.494140625, 9999.326171875, 10000.140625, 9999.9267578125, 9999.8955078125, 10001.7060546875, 10000.5419921875, 10000.4326171875, 10001.09765625, 10001.8623046875, 10001.3984375, 9998.9453125, 10000.5283203125, 10000.7734375, 10000.9638671875, 9999.859375, 10000.072265625, 9999.1142578125, 9999.6318359375, 9998.833984375, 10001.2744140625, 9999.48046875, 9999.0673828125, 10000.294921875, 10000.1064453125, 9999.3134765625, 10000.376953125, 9999.3740234375, 10000.509765625, 9999.97265625, 10000.2177734375, 9999.619140625, 9999.330078125, 10000.2548828125, 9998.4287109375, 10001.384765625, 9999.5439453125, 10002.0771484375, 10000.671875, 9999.9072265625, 9999.3974609375, 10000.51953125, 10000.462890625, 10000.4306640625, 10000.83984375, 10000.12109375, 10001.1455078125, 9999.232421875, 9999.796875, 10000.619140625, 9999.2197265625, 10000.04296875, 10001.53515625, 10000.10546875, 9999.5302734375, 10001.12890625, 9998.8642578125, 9999.66796875, 10001.0654296875, 9999.2294921875, 10000.2578125, 9999.9296875, 10001.5673828125, 9998.578125, 9999.458984375, 9999.6748046875, 10000.3740234375, 10000.5517578125, 10000.423828125, 10001.4189453125, 10000.671875, 9998.71875, 9998.5546875, 9999.462890625, 9999.474609375, 9999.5634765625, 10001.048828125, 9999.369140625, 10001.109375, 10001.1484375, 10000.03125, 9999.478515625, 10001.2880859375, 9999.154296875, 10000.478515625, 10000.2724609375, 10001.2978515625, 10000.3583984375, 10000.021484375, 10001.0546875, 9999.6201171875, 10000.5625, 10000.1328125, 10000.03515625, 9998.517578125, 9999.99609375, 9998.9892578125, 10000.5302734375, 10002.3701171875, 9999.7734375, 10000.916015625, 10001.0087890625, 9998.84765625, 10000.7080078125, 9999.6845703125, 9999.560546875, 10000.8203125, 9999.4443359375, 9999.4912109375, 10000.7578125, 10001.2900390625, 10000.06640625, 10000.6669921875, 9998.5810546875, 9999.8984375, 10000.447265625, 9999.1572265625, 10000.64453125, 10000.2822265625, 10000.2861328125, 10001.1845703125, 10001.265625, 9998.7841796875, 9999.3447265625, 10000.5380859375, 9999.69921875, 10001.990234375, 10001.10546875, 9999.1796875, 9999.244140625, 9998.734375, 10000.37109375, 9999.39453125, 9999.32421875, 9998.734375, 10000.8896484375, 9999.2529296875, 9999.09375, 9998.11328125, 10000.404296875, 10000.23046875, 10000.7314453125, 10001.435546875, 10000.20703125, 9999.6552734375, 9998.396484375, 10000.107421875, 10000.3525390625, 10000.2568359375, 9998.052734375, 10001.2353515625, 10000.62109375, 9999.6103515625, 9999.7158203125, 9998.080078125, 10000.3232421875, 9999.65625, 9998.869140625, 10001.974609375, 10001.544921875, 10000.7236328125, 9999.978515625, 9998.7587890625, 9999.98046875, 10000.4697265625, 9999.8916015625, 9999.96484375, 10000.1865234375, 9998.8564453125, 10000.8203125, 10001.0068359375, 9999.5693359375, 9999.3056640625, 10002.12109375, 10001.1416015625, 9997.6796875, 9999.8974609375, 9998.98046875, 9999.087890625, 9999.310546875, 10000.04296875, 9999.8681640625, 9998.705078125, 9998.9140625, 9997.939453125, 9999.9091796875, 10000.2685546875, 10000.861328125, 10000.9814453125, 10000.0986328125, 9999.595703125, 10002.140625, 9999.0732421875, 9998.6435546875, 9999.5380859375, 10000.1943359375, 10001.349609375, 9999.52734375, 9998.37890625, 10000.7333984375, 9999.5478515625, 9998.9306640625, 9999.0869140625, 10000.130859375, 9998.232421875, 10001.474609375, 9999.830078125, 10000.0439453125, 10001.046875, 9999.2490234375, 10000.05859375, 9999.9111328125, 10000.4765625, 10002.4423828125, 9998.8984375, 9999.927734375, 10002.1123046875, 9999.0966796875, 10001.6123046875, 10000.0771484375, 10001.0283203125, 10000.5595703125, 10000.3173828125, 9999.107421875, 9999.8505859375, 10000.68359375, 9999.61328125, 9998.9130859375, 9999.017578125, 10001.4130859375, 10001.595703125, 9999.2802734375, 9999.9775390625, 10001.2412109375, 10000.2919921875, 10000.06640625, 10000.6171875, 9998.8916015625, 9999.3154296875, 10000.7646484375, 10000.201171875, 10000.4619140625, 10000.091796875, 9998.708984375, 10000.423828125, 10001.298828125, 10000.5166015625, 10000.7333984375, 9999.0625, 9999.7509765625, 9999.5458984375, 9999.4951171875, 9999.380859375, 10000.5927734375, 10000.5537109375, 10000.845703125, 10001.8037109375, 10000.17578125, 10000.4697265625, 10000.181640625, 9998.3876953125, 9998.9423828125, 10000.470703125, 9999.5625, 9999.951171875, 9998.5771484375, 10000.2890625, 9999.705078125, 10000.1884765625, 10000.1943359375, 9998.376953125, 10001.103515625, 9998.7353515625, 9999.7021484375, 9999.5390625, 10001.0341796875, 10000.1103515625, 10000.9638671875, 10000.2060546875, 9999.978515625, 9999.1962890625, 9999.53125, 10000.0400390625, 10000.259765625, 9999.359375, 10001.4111328125, 10000.3046875, 9998.8935546875, 10001.490234375, 9998.318359375, 10000.3974609375, 9999.458984375, 10001.576171875, 10000.8388671875, 9998.837890625, 10001.0087890625, 9997.9853515625, 10000.837890625, 9999.9580078125, 9999.0927734375, 10000.9853515625, 10000.779296875, 10000.197265625, 10000.2900390625, 10000.4072265625, 9999.650390625, 10000.564453125, 9998.142578125, 9996.6220703125, 10000.4677734375, 10000.392578125, 9999.3349609375, 9998.9228515625, 10001.322265625, 9999.48828125, 9999.3974609375, 9999.033203125, 10000.7998046875, 9997.7900390625, 9999.8115234375, 9999.28515625, 9999.8564453125, 10000.6689453125, 9999.513671875, 9999.7236328125, 9998.130859375, 9999.4521484375, 9998.943359375, 10000.23046875, 9999.763671875, 9999.55859375, 10000.15625, 9999.099609375, 9998.8642578125, 9997.408203125, 9999.9228515625, 9999.9912109375, 9999.9365234375, 9999.4091796875, 10001.03515625, 9999.2431640625, 10001.7744140625, 9999.701171875, 9999.1650390625, 9999.591796875, 9999.9111328125, 10000.3642578125, 10001.859375, 10000.1220703125, 10000.6953125, 10000.90234375, 10000.4228515625, 10001.90234375, 9999.6494140625, 9999.359375, 9998.765625, 9999.7109375, 9997.880859375, 10000.60546875, 9999.28515625, 10000.6298828125, 10000.3408203125, 9998.783203125, 10002.0830078125, 9999.21484375, 9999.1357421875, 9999.572265625, 10000.458984375, 9998.908203125, 10000.7890625, 10000.1220703125, 9999.8017578125, 9998.9375, 10000.0869140625, 9999.71484375, 9997.955078125, 10000.29296875, 9999.044921875, 9999.6630859375, 9999.099609375, 9999.6259765625, 9999.517578125, 9999.595703125, 9999.5205078125, 9999.2958984375, 10000.9814453125, 10000.4150390625, 10000.5029296875, 10001.279296875, 9999.42578125, 10000.3984375, 9999.421875, 9999.11328125, 10001.271484375, 9999.1015625, 10002.4365234375, 10002.33984375, 9997.80078125, 10000.880859375, 9999.1201171875, 10000.5341796875, 10000.15625, 9997.4287109375, 10000.078125, 10001.2900390625, 10000.2216796875, 10001.8642578125, 9999.2548828125, 10001.8828125, 9999.966796875, 9998.669921875, 10000.5361328125, 10001.701171875, 10000.177734375, 10000.2197265625, 10000.103515625, 9997.8095703125, 10002.03125, 10001.9033203125, 10001.4990234375, 10000.4306640625, 10000.1015625, 9999.6845703125, 10001.5849609375, 10000.8271484375, 9998.8427734375, 9999.537109375, 10000.35546875, 9999.4521484375, 10000.705078125, 10002.0205078125, 10002.0068359375, 10000.7041015625, 9999.11328125, 10000.9873046875, 9998.7822265625, 9999.517578125, 9999.322265625, 9999.7783203125, 10000.333984375, 10001.08984375, 10000.32421875, 9998.365234375, 10000.634765625, 10000.81640625, 10000.8525390625, 9999.3369140625, 9999.8408203125, 9999.5673828125, 10000.884765625, 9998.1396484375, 9999.53515625, 10001.361328125, 9999.5322265625, 10001.3671875, 10000.826171875, 9998.107421875, 9999.130859375, 10001.9365234375, 9998.283203125, 9999.390625, 10000.1044921875, 10000.2626953125, 10001.6484375, 9999.0146484375, 10000.126953125, 10000.7646484375, 9999.68359375, 9999.0908203125, 9999.5322265625, 9998.77734375, 9999.9814453125, 10001.083984375, 9999.4755859375, 10000.666015625, 9997.966796875, 9998.1201171875, 10001.2421875, 9999.8134765625, 10000.4189453125, 9998.49609375, 9999.029296875, 10001.16796875, 9999.9404296875, 10000.626953125, 10000.0107421875, 10001.16015625, 9999.427734375, 9997.67578125, 9999.021484375, 10000.5361328125, 10000.248046875, 9999.7158203125, 10001.16796875, 9999.3759765625, 9998.8125, 10000.8603515625, 9999.2861328125, 10000.2197265625, 9998.5244140625, 9998.4853515625, 9999.3212890625, 10002.2109375, 10000.4609375, 9999.70703125, 9999.5751953125, 10000.2236328125, 10000.12890625, 10000.712890625, 10002.0380859375, 9999.666015625, 10002.654296875, 9998.0634765625, 10000.220703125, 9999.265625, 10000.5244140625, 9998.6533203125, 9999.166015625, 10000.0302734375, 9998.6201171875, 10003.1474609375, 10001.1689453125, 10000.828125, 10000.39453125, 10000.5087890625, 9999.853515625, 10000.2509765625, 9999.76171875, 9998.94140625, 9999.8115234375, 10001.4599609375, 10000.330078125, 10001.6435546875, 10001.021484375, 10002.294921875, 10000.2333984375, 10000.177734375, 9999.5927734375, 10001.2373046875, 9999.580078125, 10000.6630859375, 9998.2939453125, 9999.54296875, 10001.4580078125, 10000.53515625, 10001.37109375, 9999.4931640625, 9999.787109375, 9998.3154296875, 10000.0009765625, 10000.8408203125, 9999.78125, 10000.10546875, 9999.052734375, 9999.1513671875, 10000.0166015625, 10000.7900390625, 9999.642578125, 10000.716796875, 10001.171875, 9999.52734375, 9998.845703125, 9997.748046875, 10000.2509765625, 9999.26953125, 10000.0908203125, 9999.3291015625, 9999.751953125, 10000.38671875, 9999.63671875, 10001.029296875, 9999.2099609375, 9999.36328125, 9999.5908203125, 10000.43359375, 9998.90234375, 10001.15234375, 9998.20703125, 9999.8251953125, 10000.3466796875, 9998.490234375, 10000.3662109375, 10000.8291015625, 10000.234375, 9999.931640625, 9999.91015625, 9999.9794921875, 10000.9541015625, 9999.736328125, 9999.5068359375, 10000.8701171875, 10000.5927734375, 10001.0546875, 9998.8623046875, 10001.1171875, 9999.8271484375, 9999.0732421875, 10000.767578125, 10000.3330078125, 9998.916015625, 9999.7646484375, 10000.638671875, 9998.2275390625, 10000.5458984375, 9999.791015625, 10000.5146484375, 9997.7724609375, 10001.5986328125, 9999.3876953125, 9999.4150390625, 9999.759765625, 10000.548828125, 9999.857421875, 10001.5537109375, 9999.7646484375, 10001.64453125, 9999.796875, 10000.0068359375, 9998.408203125, 9999.1943359375, 10000.294921875, 9998.5458984375, 10001.689453125, 10000.21484375, 9999.642578125, 9999.91015625, 10001.673828125, 10000.7822265625, 9998.099609375, 9999.5810546875, 10000.9189453125, 9999.1484375, 10000.357421875, 9997.7265625, 9999.4111328125, 10000.1103515625, 10000.7265625, 9999.517578125, 10000.9267578125, 9998.5107421875, 9999.8134765625, 10000.1845703125, 9998.427734375, 10001.29296875, 10000.2919921875, 9998.892578125, 10000.5537109375, 9999.1630859375, 9999.34765625, 9998.62890625, 9999.638671875, 9997.80078125, 10000.2158203125, 10001.8525390625, 9999.5537109375, 9997.6943359375, 9999.51953125, 9999.9296875, 10000.1513671875, 9999.93359375, 10001.5224609375, 9999.8115234375, 9999.603515625, 10000.3486328125, 10000.4443359375, 9999.7861328125, 9999.755859375, 9999.3486328125, 10000.7294921875, 9998.4443359375, 10000.1162109375, 10000.8876953125, 9999.943359375, 9998.2578125, 9999.8349609375, 10000.7265625, 10000.2275390625, 9998.916015625, 10002.28515625, 10000.9306640625, 10000.15234375, 10002.7724609375, 9999.70703125, 9998.4873046875, 9999.8330078125, 10000.0673828125, 9998.4267578125, 10000.69140625, 10000.9853515625, 9998.576171875, 9999.3486328125, 9998.7744140625, 9998.6943359375, 10000.4404296875, 10000.08984375, 9999.7783203125, 10000.3076171875, 9999.611328125, 9999.34765625, 9999.6005859375, 9999.75, 10000.017578125, 10000.5107421875, 9999.2666015625, 9999.7060546875, 9999.4228515625, 9999.373046875, 10000.7900390625, 9999.2900390625, 9999.8447265625, 9998.9775390625, 10000.703125, 10001.1005859375, 10000.169921875, 10001.9365234375, 9997.8916015625, 9999.3056640625, 10000.6552734375, 9998.44140625, 9999.8310546875, 10000.0712890625, 9999.7509765625, 9999.9541015625, 10000.849609375, 10001.0830078125, 9998.5888671875, 9998.677734375, 10000.4443359375, 9998.8232421875, 10001.91015625, 9999.896484375, 10000.23046875, 9999.8232421875, 9999.6318359375, 9999.267578125, 10001.6884765625, 10000.3642578125, 9997.982421875, 9999.36328125, 10000.5302734375, 10000.419921875, 10001.228515625, 10000.2685546875, 9999.7451171875, 9999.8251953125, 10000.291015625, 10000.5439453125, 9999.3603515625, 9999.673828125, 10000.5693359375, 10000.1435546875, 9999.8955078125, 10000.8388671875, 9999.2587890625, 10000.6044921875, 10001.2236328125, 9999.9931640625, 9999.91015625, 9998.9423828125, 9999.6328125, 9999.3388671875, 10001.0, 9999.7392578125, 9999.71484375, 10001.0810546875, 9999.9228515625, 10000.5517578125, 9999.640625, 10000.38671875, 9999.82421875, 10001.2861328125, 10000.2314453125, 9998.5966796875, 9999.5126953125, 9998.736328125, 10000.5390625, 10000.7861328125, 10000.2568359375, 9999.4951171875, 9998.861328125, 9999.9541015625, 9999.3330078125, 10000.9296875, 9999.6103515625, 10000.45703125, 9999.5205078125, 10001.09375, 9998.984375, 9999.6455078125, 10000.580078125, 10001.67578125, 9998.568359375, 9998.728515625, 10000.470703125, 10001.5791015625, 10000.83984375, 10000.716796875, 9998.9326171875, 10000.26171875, 10000.5107421875, 10000.212890625, 10000.3046875, 10000.75390625, 9997.564453125, 9997.962890625, 9998.810546875, 10000.0859375, 9999.6103515625, 9998.666015625, 10001.6201171875, 9999.7919921875, 10000.1962890625, 10000.93359375, 10002.1298828125, 9999.9404296875, 10000.185546875, 9998.7548828125, 9999.9521484375, 9998.7392578125, 9999.546875, 10000.76171875, 10000.357421875, 10000.947265625, 10000.2509765625, 10001.166015625, 9999.017578125, 10001.216796875, 9999.427734375, 9999.6220703125, 9999.6865234375, 10001.83203125, 9999.2451171875, 10000.4814453125, 10000.41796875, 10000.0966796875, 9999.208984375, 9998.853515625, 9998.84765625, 10001.0869140625, 9999.208984375, 9998.787109375, 10000.9541015625, 10000.61328125, 9999.9619140625, 9999.466796875, 9997.2587890625, 9999.212890625, 9999.8193359375, 9998.916015625, 9998.9169921875, 10002.044921875, 9998.2822265625, 10000.28125, 9999.1826171875, 10000.1728515625, 9998.765625, 10001.17578125, 9999.6416015625, 10001.6865234375, 10001.3447265625, 9999.0654296875, 9998.9677734375, 9998.6650390625, 9998.98046875, 10001.2080078125, 10001.6064453125, 9999.384765625, 10001.0703125, 9999.9287109375, 9999.70703125, 9999.8330078125, 10000.7744140625, 9997.9677734375, 9999.61328125, 9999.3662109375, 10000.8037109375, 10000.734375, 10001.001953125, 10000.6328125, 10000.349609375, 9999.1708984375, 10000.2568359375, 9999.6240234375, 9999.4697265625, 9997.765625, 10001.0810546875, 9999.021484375, 9999.9638671875, 10001.8984375, 10000.908203125, 9999.5263671875, 10002.5126953125, 10001.05078125, 9999.1591796875, 9999.5439453125, 10001.05078125, 10000.8515625, 10000.416015625, 9998.7939453125, 9999.91015625, 10000.8603515625, 10001.916015625, 10000.1728515625, 9999.8115234375, 9999.791015625, 10000.5234375, 9998.1884765625, 9999.6015625, 9999.8818359375, 10000.3671875, 9999.939453125, 9999.7578125, 9999.5029296875, 10000.0810546875, 10000.5341796875, 10000.3916015625, 10001.0, 9999.5859375, 10000.6220703125, 9998.48046875, 9999.6787109375, 10001.435546875, 9999.119140625, 10000.7158203125, 9999.3779296875, 10000.1396484375, 9998.46875, 10001.7763671875, 10001.525390625, 9999.29296875, 9998.990234375, 9998.3984375, 10000.2451171875, 10000.2275390625, 10000.6025390625, 10000.4677734375, 9999.04296875, 10001.1064453125, 10001.5986328125, 9999.7197265625, 9999.44140625, 9999.80859375, 9999.6396484375, 10000.904296875, 10000.9365234375, 10000.6513671875, 9999.7197265625, 10000.078125, 10001.0498046875, 10000.6572265625, 10000.2431640625, 10000.37890625, 10001.17578125, 9999.6611328125, 9999.0087890625, 9999.6806640625, 10001.92578125, 9999.232421875, 10001.25390625, 9998.9287109375, 10000.0625, 10000.697265625, 9999.65234375, 9999.3935546875, 10000.44140625, 9998.9892578125, 10001.1171875, 9999.439453125, 10000.5185546875, 9999.4404296875, 9999.9072265625, 9999.88671875, 10001.009765625, 10000.6142578125, 9998.8740234375, 9998.58203125, 9998.845703125, 9999.185546875, 10001.0712890625, 10000.181640625, 10001.115234375, 9999.228515625, 9999.7353515625, 9999.572265625, 9998.46875, 10001.2197265625, 9998.9267578125, 9998.7705078125, 10000.30859375, 10001.04296875, 10000.8173828125, 10000.5234375, 10001.5341796875, 9999.4130859375, 9998.330078125, 10000.560546875, 9999.658203125, 10000.6396484375, 10001.923828125, 10001.0849609375, 10000.353515625, 9999.458984375, 10001.1259765625, 10001.0908203125, 10000.8369140625, 10000.2890625, 9999.1298828125, 10000.3076171875, 10001.30859375, 9999.3427734375, 10000.6728515625, 10002.052734375, 10000.7265625, 10000.66015625, 10000.2216796875, 10000.8037109375, 10003.1435546875, 9999.3232421875, 9999.3720703125, 10000.37890625, 9999.056640625, 9999.234375, 9999.068359375, 10000.0546875, 9998.0009765625, 9999.3388671875, 9998.8701171875, 9999.6025390625, 10000.470703125, 10000.466796875, 10000.1044921875, 10000.5556640625, 9999.7041015625, 9999.9228515625, 9999.2421875, 10000.15234375, 9999.6630859375, 9998.33203125, 10000.0244140625, 10002.248046875, 9998.43359375, 10000.0654296875, 9998.2099609375, 10001.55859375, 10001.869140625, 10001.0771484375, 10001.484375, 9999.8203125, 9999.3046875, 9998.6396484375, 9999.9404296875, 10000.8955078125, 10000.439453125, 9999.763671875, 10001.6865234375, 9998.712890625, 9999.1650390625, 10000.935546875, 9998.0908203125, 9998.7109375, 9999.8623046875, 10000.6015625, 10000.607421875, 10001.1865234375, 10000.52734375, 9999.0498046875, 9999.5927734375, 9999.5341796875, 10000.80859375, 10000.697265625, 9999.5498046875, 9998.318359375, 10000.177734375, 9998.48828125, 9998.91796875, 10000.1455078125, 10000.2392578125, 9999.228515625, 10000.0, 10000.32421875, 10001.734375, 10000.7255859375, 9999.2470703125, 9999.5263671875, 10000.2919921875, 10000.8037109375, 9999.9658203125, 9999.8740234375, 10001.4150390625, 10000.017578125, 9997.998046875, 10002.734375, 9999.6494140625, 9999.513671875, 10000.3447265625, 10000.099609375, 10000.818359375, 10000.080078125, 10000.6533203125, 9999.2158203125, 9997.6572265625, 10000.365234375, 9998.9423828125, 9998.416015625, 10000.140625, 10000.115234375, 10000.47265625, 10001.0478515625, 9999.8017578125, 9998.3740234375, 10000.7646484375, 10000.5986328125, 9998.9189453125, 10001.873046875, 9999.1748046875, 10000.998046875, 10000.6904296875, 9998.984375, 9999.5078125, 10000.6005859375, 10001.0380859375, 9999.2919921875, 9999.345703125, 9998.71484375, 10001.279296875, 10000.8857421875, 9999.876953125, 9998.7060546875, 9999.8203125, 9999.9873046875, 10000.4697265625, 9998.1806640625, 10000.7001953125, 9999.712890625, 10000.9716796875, 10000.5478515625, 9998.96875, 9999.990234375, 9999.5029296875, 10000.1767578125, 9999.16796875, 9999.529296875, 9999.5029296875, 9998.1884765625, 10000.447265625, 10001.8671875, 9999.5712890625, 10000.4189453125, 9999.587890625, 9997.5166015625, 9999.017578125, 10001.6044921875, 10002.1904296875, 9999.9892578125, 9999.1591796875, 10000.7919921875, 9999.0615234375, 10000.2822265625, 10000.0078125, 9999.220703125, 10000.591796875, 9999.9345703125, 10001.7412109375, 10001.1259765625, 10000.0771484375, 9998.7060546875, 9999.0146484375, 10001.056640625, 10000.7998046875, 9998.8701171875, 10000.05078125, 9999.228515625, 9998.8583984375, 9998.287109375, 9999.859375, 9999.9443359375, 9999.5986328125, 9998.744140625, 10000.2783203125, 9999.423828125, 10000.681640625, 10000.212890625, 9999.5419921875, 10000.900390625, 9999.248046875, 9999.298828125, 10000.0517578125, 10000.7333984375, 9999.810546875, 10000.111328125, 10000.265625, 9999.6748046875, 10000.25390625, 10000.2177734375, 9998.3232421875, 10001.2998046875, 10001.447265625, 10000.1513671875, 10001.0361328125, 10000.14453125, 10000.7978515625, 10001.0078125, 9999.599609375, 10000.48046875, 10001.39453125, 10001.10546875, 10001.55078125, 10001.4765625, 10000.6259765625, 9999.095703125, 10000.6455078125, 10000.953125, 10000.892578125, 9999.453125, 10000.7880859375, 9999.4873046875, 10000.765625, 10001.96484375, 9999.087890625, 9999.6357421875, 10000.10546875, 10001.3271484375, 10003.3115234375, 10000.517578125, 10000.1396484375, 9999.8759765625, 9998.1708984375, 10000.021484375, 10000.1748046875, 10001.3974609375, 9997.947265625, 9999.9365234375, 10000.236328125, 9998.7685546875, 10001.2509765625, 10000.1787109375, 10000.611328125, 9999.8544921875, 9999.671875, 9998.384765625, 10000.705078125, 10001.0419921875, 10000.6064453125, 9999.1904296875, 9999.353515625, 9998.73828125, 9999.7744140625, 10001.4072265625, 10000.65625, 9999.3017578125, 9998.5263671875, 10000.2548828125, 10000.4609375, 9999.7685546875, 9999.6416015625, 9999.4609375, 9999.767578125, 9999.0224609375, 10000.2333984375, 10000.4130859375, 10000.2548828125, 10000.1220703125, 10001.296875, 9999.7236328125, 10000.0166015625, 10001.7626953125, 9998.5791015625, 9997.974609375, 10000.5029296875, 9999.7763671875, 10001.908203125, 10000.962890625, 10000.755859375, 10000.8115234375, 9998.9765625, 10000.0986328125, 10002.05078125, 9998.748046875, 9999.9580078125, 9998.95703125, 10000.6005859375, 10000.9130859375, 9999.06640625, 9999.986328125, 10001.3486328125, 10000.0810546875, 9998.95703125, 10000.4501953125, 10001.9541015625, 9998.6181640625, 9999.5517578125, 9999.705078125, 10000.5029296875, 10000.2900390625, 10000.580078125, 10001.8544921875, 9999.94921875, 9999.1572265625, 10000.0849609375, 10001.501953125, 9999.8212890625, 10000.6708984375, 10000.1396484375, 9999.8134765625, 9999.5888671875, 10000.904296875, 9999.4892578125, 9999.794921875, 9998.306640625, 9999.7705078125, 9999.8037109375, 10000.123046875, 9999.4443359375, 9998.6826171875, 10000.326171875, 10000.5322265625, 9999.2802734375, 10000.197265625, 9999.49609375, 9999.7744140625, 10000.0888671875, 9999.4033203125, 9998.9423828125, 10001.859375, 9999.7861328125, 10000.173828125, 10001.2607421875, 10001.0673828125, 9998.9912109375, 9999.466796875, 10002.8994140625, 10000.0556640625, 10000.7958984375, 10000.2705078125, 9999.75, 10000.0283203125, 10000.248046875, 10000.3173828125, 9997.37890625, 9999.4599609375, 9999.1865234375, 9998.12109375, 9999.0615234375, 9997.76171875, 10001.751953125, 9999.466796875, 9999.4609375, 10000.708984375, 9999.7236328125, 10000.6181640625, 9999.6201171875, 9999.87109375, 9999.6103515625, 10000.833984375, 9999.408203125, 9997.4482421875, 10000.0224609375, 10000.935546875, 9999.2353515625, 9998.892578125, 9999.345703125, 9997.6298828125, 9998.9189453125, 9999.1943359375, 10000.6572265625, 9999.484375, 10000.673828125, 9998.5966796875, 10002.4365234375, 9999.08203125, 9998.9130859375, 10001.2705078125, 9998.8359375, 10000.2314453125, 10002.783203125, 9999.42578125, 9998.0205078125, 10000.0009765625, 9999.27734375, 9999.017578125, 10000.466796875, 10000.9697265625, 10000.365234375, 10000.3017578125, 9998.4345703125, 9999.4189453125, 10000.7578125, 9998.830078125, 10000.3974609375, 9999.0244140625, 10000.8447265625, 9999.03125, 10000.716796875, 10000.7138671875, 9999.978515625, 9999.7685546875, 9998.6875, 9998.5830078125, 10002.0693359375, 9999.1025390625, 9998.009765625, 10000.2333984375, 9999.4140625, 10000.033203125, 9999.9140625, 9998.71484375, 9997.88671875, 9999.8740234375, 10001.11328125, 9999.5458984375, 10000.154296875, 10000.6845703125, 9999.330078125, 9999.140625, 9999.3515625, 10000.7958984375, 10000.419921875, 10000.185546875, 9998.6630859375, 10001.8193359375, 9999.9912109375, 9999.67578125, 10000.6025390625, 10000.6025390625, 9999.818359375, 10000.59375, 10001.8857421875, 10000.5185546875, 9999.71484375, 9998.87890625, 9998.865234375, 9999.14453125, 9999.3193359375, 9998.6923828125, 9999.6435546875, 9999.7861328125, 9998.890625, 9998.2138671875, 10000.8515625, 10000.673828125, 9999.611328125, 9999.97265625, 10000.0830078125, 10000.6513671875, 10000.5, 9999.8017578125, 9999.4921875, 9999.177734375, 10000.4755859375, 10001.7373046875, 10000.041015625, 10000.740234375, 10001.0849609375, 9999.9013671875, 9998.1015625, 9999.2939453125, 10000.9677734375, 9999.7705078125, 10001.08203125, 9999.10546875, 10000.5126953125, 9999.21484375, 9999.267578125, 10000.6435546875, 10000.599609375, 9998.5888671875, 10000.708984375, 10000.0634765625, 10000.658203125, 10000.73828125, 10000.20703125, 10000.98828125, 9999.5537109375, 10000.9189453125, 9997.8154296875, 9999.837890625, 9999.3642578125, 10000.0400390625, 10001.40625, 10001.37109375, 10001.0126953125, 10000.115234375, 9998.1328125, 10000.3134765625, 10000.23046875, 9999.498046875, 9999.49609375, 10001.7021484375, 10001.1494140625, 9999.546875, 10000.84765625, 10000.484375, 9999.9169921875, 9999.2802734375, 10000.103515625, 9999.7822265625, 9998.119140625, 9999.9443359375, 10001.6630859375, 9998.7890625, 10000.546875, 10000.8046875, 9999.4482421875, 10000.6533203125, 10002.060546875, 10000.4326171875, 10000.16015625, 10000.029296875, 10001.03515625, 10000.44921875, 10000.580078125, 9999.3876953125, 9999.5751953125, 10000.896484375, 10002.244140625, 10001.2578125, 10000.802734375, 9999.220703125, 10000.5712890625, 9999.427734375, 10001.29296875, 9999.53515625, 9997.8916015625, 10002.6083984375, 10000.4208984375, 9999.822265625, 10000.50390625, 9998.3359375, 9999.6484375, 9999.0498046875, 9999.05859375, 9999.2255859375, 10000.755859375, 10000.1806640625, 9999.95703125, 9999.365234375, 9999.541015625, 9999.513671875, 10000.3271484375, 10000.63671875, 10000.080078125, 9999.912109375, 10000.822265625, 10000.935546875, 9999.6337890625, 9999.7744140625, 10000.740234375, 10001.4326171875, 10000.9501953125, 10000.640625, 10000.40234375, 9998.9248046875, 9999.12890625, 10000.2099609375, 9999.6953125, 10000.4765625, 10000.4169921875, 10000.9111328125, 9999.0810546875, 10000.220703125, 9998.376953125, 9999.6591796875, 10001.013671875, 9999.06640625, 9999.8291015625, 9998.857421875, 10000.25, 10002.0595703125, 9998.7216796875, 9999.626953125, 10000.642578125, 10001.0966796875, 10000.43359375, 10000.861328125, 9999.15625, 10000.1884765625, 9999.7529296875, 9999.0693359375, 9999.8779296875, 9999.224609375, 9998.958984375, 10000.1796875, 10001.91015625, 9998.703125, 9998.8115234375, 10000.255859375, 9998.42578125, 9999.22265625, 10000.505859375, 9999.3671875, 9999.9677734375, 10001.6279296875, 10000.599609375, 9998.5927734375, 9999.3955078125, 10001.736328125, 9999.9013671875, 9998.666015625, 10000.4990234375, 10000.51953125, 10001.841796875, 10000.4453125, 10000.8359375, 10001.3154296875, 9998.416015625, 9999.87890625, 9999.7587890625, 9999.90234375, 9999.814453125, 9999.5556640625, 10000.9150390625, 9999.513671875, 10001.3330078125, 9999.0244140625, 9999.37890625, 9997.771484375, 9998.4814453125, 10001.9140625, 9998.7763671875, 10000.0947265625, 10000.6787109375, 10000.416015625, 10001.404296875, 10002.375, 9999.7744140625, 9999.6904296875, 10001.05078125, 9999.330078125, 9998.3271484375, 10000.3125, 9999.3193359375, 9999.51953125, 9999.3466796875, 9999.7373046875, 10000.2626953125, 9999.052734375, 10003.0126953125, 9997.6943359375, 10001.326171875, 9999.6865234375, 9999.267578125, 9999.759765625, 10000.9833984375, 9999.6044921875, 9999.865234375, 9999.748046875, 9999.1201171875, 10001.099609375, 10000.6513671875, 10000.197265625, 10001.0146484375, 10000.9306640625, 10000.30078125, 9999.763671875, 9999.9853515625, 10000.5458984375, 10001.19921875, 10000.013671875, 10000.720703125, 10000.619140625, 9998.5048828125, 9998.6865234375, 10001.376953125, 9999.646484375, 9999.09765625, 10001.4736328125, 9998.625, 10001.23828125, 10000.681640625, 10000.2763671875, 9999.076171875, 10000.7607421875, 9999.236328125, 10000.541015625, 10000.193359375, 10000.2158203125, 9999.0537109375, 9998.6806640625, 9999.7509765625, 10000.5859375, 9999.6171875, 9998.51953125, 10000.404296875, 10000.10546875, 10000.0634765625, 9999.9091796875, 9998.705078125, 9999.2841796875, 9999.857421875, 10000.767578125, 10000.2373046875, 10000.173828125, 10002.1083984375, 10000.294921875, 9999.21875, 10000.2041015625, 9997.37109375, 9999.93359375, 9999.794921875, 9998.4287109375, 9998.451171875, 10000.73046875, 9999.70703125, 9999.4453125, 10000.572265625, 10000.2744140625, 10000.7529296875, 9999.412109375, 10000.1220703125, 10002.0634765625, 10000.41015625, 9999.6796875, 10000.7919921875, 10000.6572265625, 10001.9326171875, 9999.8115234375, 10000.271484375, 10000.90625, 10000.447265625, 9999.6787109375, 10001.2880859375, 9997.9384765625, 10000.0986328125, 10000.3193359375, 9998.7490234375, 10000.4296875, 9999.6806640625, 9999.2587890625, 9999.5947265625, 10001.7646484375, 10000.53515625, 9997.88671875, 10000.423828125, 10000.0849609375, 9999.47265625, 9999.837890625, 9999.05859375, 10000.39453125, 9999.431640625, 9999.708984375, 9998.9658203125, 10001.48046875, 9997.5751953125, 9999.669921875, 9996.9296875, 10000.2275390625, 9998.8935546875, 10000.3828125, 9999.2470703125, 10001.884765625, 10001.5771484375, 9999.2177734375, 10001.2509765625, 10000.595703125, 10000.259765625, 9999.08984375, 9999.7197265625, 10000.3603515625, 10001.2138671875, 9999.095703125, 10000.2119140625, 9998.5908203125, 10000.81640625, 10000.4951171875, 10000.34375, 10000.3427734375, 9999.4501953125, 10000.53515625, 10000.3349609375, 9999.537109375, 9999.3857421875, 10001.0673828125, 9999.46875, 10000.603515625, 9999.3232421875, 9999.3779296875, 10000.666015625, 10001.93359375, 10000.2255859375, 10000.0, 10000.697265625, 9999.17578125, 10000.1923828125, 9999.513671875, 9998.5419921875, 10001.6904296875, 9999.66796875, 9999.322265625, 9999.615234375, 9999.02734375, 9999.6953125, 9997.2451171875, 10000.27734375, 9998.3310546875, 9998.04296875, 10001.2626953125, 10000.6396484375, 9998.6845703125, 10001.138671875, 10000.453125, 9999.8818359375, 9999.8623046875], "dtype": "float32"}], "e": {"__tup__": [9999.9921875, 0.9892677664756775]}}, {"a": [{"__nd__": [-0.0009846125030890107, -0.0004422346828505397, 0.0022417735308408737, -0.00011740717309294268, -0.0007260189740918577, 0.0005124846356920898, 0.00024255683820229024, -0.0010288972407579422, -0.0003836694231722504, -0.0011577022960409522, 0.0018116150749847293, 0.0005266350344754755, -0.0010100840590894222, -8.79870810877037e-07, 0.001494777505286038, 0.0002932656498160213, 1.9138849893352017e-05, 0.00031178948120214045, -0.001042752293869853, 0.0005177802522666752, -0.0012464913306757808, 0.00037592812441289425, 0.0014297685120254755, -0.00014137770631350577, 0.0005914096254855394, -0.0006484798504970968, 0.0003883561003021896, -0.0010002875933423638, -0.002517662476748228, 0.0001432929129805416, 0.0006270603626035154, -0.0003417172993067652, -0.000709099171217531, -0.00011973840446444228, 0.00046291365288197994, 0.0014925375580787659, 0.0009140401962213218, 0.0006353827193379402, -0.0008157319389283657, -0.0006195491296239197, -0.0025107155088335276, -0.00016914094157982618, 0.00020415702601894736, 0.0017355424351990223, -0.002956119831651449, -0.0003561236953828484, -0.0014309616526588798, -0.00014149630442261696, -0.000746020523365587, 0.0022726869210600853, 0.0013434231514111161, 0.00019062214414589107, 0.00010856206790776923, -0.002335340715944767, 1.4155662938719615e-05, -0.0004291015793569386, 0.0006987318047322333, -0.0007399562164209783, -0.0011106297606602311, 0.0011255687568336725, -0.0006847163895145059, 0.00021462164295371622, 0.0005907929735258222, 0.0019473180873319507, 4.0346363675780594e-05, 0.00027924354071728885, 0.0009329707245342433, 0.001213138923048973, -0.0009172232239507139, 0.00015347168664447963, 0.0014190392103046179, -0.0014793748268857598, 0.0006050972733646631, -0.003372201696038246, -6.0506605223054066e-05, -0.0005318929324857891, 0.0005245169159024954, 0.00025585602270439267, -8.195459668058902e-05, 0.0009311040630564094, -0.0006008832133375108, -0.001065094256773591, -0.001679147477261722, 0.00010089742136187851, 0.002174730645492673, 0.0006654133321717381, 0.00016405498899985105, -0.0020524896681308746, 0.0010659564286470413, 0.001439664396457374, -0.0005694697028957307, -0.0011395005276426673, -0.0015418992843478918, -0.0015904353931546211, 0.0002780697541311383, 0.0009338190429843962, 0.0003371547791175544, 0.0015711470041424036, -3.263936014263891e-05, -0.00034503446659073234, -0.00034926942316815257, -0.00010403183114249259, 0.0022029546089470387, 0.000600170053075999, 0.000793426763266325, 0.0015599089674651623, -0.0007766573107801378, 0.002315943129360676, -0.0009565203217789531, 0.0007783269393257797, 0.001488670357502997, 0.00020930735627189279, 0.00057714938884601, -0.0003895280242431909, -0.0004682607832364738, 0.00022356414410751313, -0.00029651212389580905, -9.094255074160174e-05, 0.0002940229023806751, 0.0006558542954735458, 0.00016763526946306229, -0.0011275737779214978, -0.0001202154453494586, -0.0006962504121474922, 0.0002968413464259356, 0.00069485540734604, 0.0004897501785308123, 0.00016446957306470722, 0.00042986051994375885, -0.0006421554717235267, 0.001599412877112627, 0.00115468620788306, -0.0007333735120482743, -0.00014421666855923831, 0.000451644096756354, 0.0004324746842030436, 1.454511220799759e-05, -0.0011184249306097627, 0.0010375308338552713, 0.0004923875676468015, -0.00028862542239949107, -0.00037619718932546675, -0.0009987441590055823, -0.0016147007700055838, 8.137713302858174e-05, 0.0015650450950488448, 0.0009675899054855108, 0.00023018741921987385, 0.0009681344381533563, 1.0865440344787203e-05, -0.0014622025191783905, -0.001996851759031415, 0.0010740485740825534, -0.0010714419186115265, 0.0007966464036144316, -9.624770609661937e-05, 0.00044972661999054253, -6.145329098217189e-05, 0.0015988530358299613, 0.00016374056576751173, -0.0002768950071185827, 0.0008930727490223944, 0.0011413745814934373, 0.00031989836134016514, -0.0005495325895026326, -0.001972429919987917, 0.0004306643968448043, -0.0005313632427714765, -0.0003942167677450925, 0.0004477643524296582, 0.00043707111035473645, 0.0019067837856709957, -4.782847099704668e-05, -0.0011533068027347326, 0.0008923170389607549, 0.0016098051564767957, 0.0002044803841272369, -0.0017738330643624067, -0.0009917806601151824, 0.0003629468847066164, 0.0020693009719252586, 0.00088145473273471, 0.0009808761533349752, 0.0003040062729269266, -0.0020948550663888454, 0.00021537981228902936, -0.00032710094819776714, -0.0018487808993086219, 0.0004659860278479755, 0.000347770779626444, -0.0002509657060727477, 6.182458309922367e-05, 0.0009639939526095986, -0.001684943912550807, -0.00011794660531450063, -0.0012377123348414898, 0.0005689893150702119, -0.0019197989022359252, 0.00040565390372648835, -0.00031458589364774525, -0.0006154211587272584, 0.0008289554971270263, 0.001212162314914167, 0.0009903049794957042, -8.260749382316135e-06, 0.001995340920984745, -0.00029041775269433856, 0.0008470393368043005, -0.00018383313727099448, -0.0013628575252369046, 0.0003700133820530027, -0.00046879088040441275, 0.00047633066424168646, 0.0013618270168080926, 0.0010836503934115171, 0.0010731663787737489, 0.0017803263617679477, -0.0005941675626672804, -0.0005130202625878155, 0.0011262099724262953, 0.0008437495562247932, -0.0007126931450329721, -0.0008167803753167391, 0.00011483882553875446, 0.0015636659227311611, -0.00094215216813609, -0.0006880605360493064, -0.0006534768035635352, -0.0019231276819482446, 0.0014240409946069121, 0.0008328628609888256, -0.0008708368404768407, 0.0003641338844317943, 0.0005804203101433814, 1.6606185454293154e-05, -0.0012924420880153775, 0.0014723723288625479, -9.535676508676261e-05, 0.0010870440164580941, -0.0012566499644890428, -0.0019735845271497965, 0.0009761896799318492, -0.0015938886208459735, 0.00017227965872734785, -0.0009143671486526728, 0.0004317673738114536, -0.0013054109876975417, 0.0005540335550904274, 0.0023186092730611563, -4.675767286244081e-06, 0.0014096293598413467, 0.001525977859273553, -0.00043229106813669205, 0.0007316336268559098, -0.00021277648920658976, -0.00040990536217577755, 0.0007052337168715894, 0.0002571140939835459, 0.0006130722467787564, -0.0003572376735974103, 0.0004995110211893916, 0.0009793585631996393, 0.00015271161100827157, 0.0006539870519191027, -0.0019941499922424555, 0.0004202614654786885, -0.002450966741889715, -0.0006712584290653467, 0.00019517431792337447, -0.00036356711643747985, -0.0013549239374697208, 0.0004429884138517082, 0.00103256874717772, -0.0004711601068265736, 0.0007000911864452064, 0.0015908889472484589, -0.0006340576219372451, -0.0007886511157266796, 0.0017851818120107055, 0.0015086447820067406, 0.0006653882446698844, -0.001658648019656539, 0.0023394825402647257, 7.037437171675265e-05, 0.0007056159083731472, -5.24547285749577e-05, -0.0003523883642628789, -0.0014113219222053885, -0.0011667541693896055, -0.0008407763671129942, -0.0008146672043949366, -0.0013549806317314506, -0.00028243145789019763, 0.0008362248772755265, -0.00030350525048561394, 1.9012213670066558e-05, 0.0017199176363646984, -0.00248075183480978, 0.00024001103884074837, 0.0008510888437740505, 0.00012607169628608972, -0.002059245714917779, -0.0007877221796661615, 0.00010737801494542509, -0.0005280165350995958, 0.00042891380144283175, -0.0004283679008949548, -0.0011087182210758328, -0.0008033351041376591, -0.001256826682947576, -0.0001554561749799177, 0.000690227490849793, -0.00036546928458847106, 0.00035354544525034726, -0.0005411552847363055, 3.230404036003165e-05, 6.443886377383024e-05, -0.0017139361007139087, 0.0009754146449267864, 0.001280750846490264, -0.0008269845275208354, -0.0006927925860509276, -0.000466719560790807, -0.001456527505069971, -0.0011107934406027198, -0.0005484502762556076, -0.0016385847702622414, -0.0009284619591198862, 0.0008495269576087594, -0.0002967208274640143, -0.0006702131358906627, -0.00024983842740766704, 0.0015919668367132545, 0.0007027507526800036, -0.0017407790292054415, 0.00031771158683113754, 0.00029560833354480565, -0.0015683419769629836, 0.0010177040239796042, -0.00011340484343236312, -0.0011344028171151876, -0.0001249392662430182, -0.0012832821812480688, 0.0005392302991822362, -0.0005620330921374261, -0.0007919986383058131, -0.0008623457397334278, 0.00011696288856910542, 0.00022496585734188557, -0.0009501521708443761, -0.0022478243336081505, -0.0007563572726212442, -0.0004623176937457174, 0.00011420906957937405, -0.0003580271149985492, 0.00018064590403810143, 0.0009140375186689198, -0.0007228456670418382, -0.0004465638776309788, -0.0005187138449400663, -0.0006237432826310396, 0.0002946748863905668, 0.00027975410921499133, 0.0015002641594037414, 0.00046971553820185363, -0.00017215356638189405, -0.0014078451786190271, 0.001060744863934815, -0.0003864819591399282, 5.573103862843709e-06, 0.002505442127585411, -0.0003810384077951312, 0.0005746882525272667, -0.00028643288533203304, 0.0009924854384735227, 0.0006244928226806223, 0.0010457515018060803, -0.00114873715210706, 0.0004858097527176142, 0.0002446488942950964, 0.0004652954521588981, -0.001618480309844017, 0.00017256276623811573, 0.0002810826408676803, 0.0007908728439360857, 0.0010062453802675009, -0.0010570932645350695, -0.001562087913043797, 0.001820148783735931, -0.0006931436364538968, 0.0013194045750424266, 0.0014841906959190965, 0.0008701301412656903, 0.0012130922405049205, -0.00026351638371124864, 0.00011550579074537382, 0.0008024688577279449, -0.0005478390958160162, -0.0007289142231456935, -0.00040191400330513716, -0.0018426551250740886, 0.0006284084520302713, -7.363769691437483e-05, 0.0015165586955845356, 0.00043196367914788425, 0.001711767166852951, 0.0009945627534762025, 0.0005235229036770761, 0.0012572031700983644, -0.0013338117860257626, 0.0010519498027861118, -5.387794954003766e-05, -0.0011401032097637653, -0.0011796840699389577, 0.0008252250263467431, 0.0002462730335537344, -0.00045830034650862217, -0.0021821032278239727, -7.86273303674534e-05, 0.000529965793248266, -0.0010886419331654906, -0.0005934166838414967, -0.0009805434383451939, 0.0010711143258959055, 1.3923315236752387e-05, -0.00042069671326316893, -0.0011562580475583673, 0.0018437565304338932, -0.0006674843607470393, 0.0008548171608708799, 0.000969369022641331, -0.00032284078770317137, 0.0005985014722682536, -4.220507253194228e-05, -0.0009270237642340362, -0.000802300579380244, -0.00012301230162847787, 0.0012086089700460434, -0.00034116231836378574, 0.00011648402869468555, 0.0009293791954405606, 0.001075040316209197, -0.0012132229749113321, 0.002690047724172473, 0.0009885206818580627, -0.0003108466335106641, 0.0004161643737461418, 0.0013645841972902417, -0.000737344438675791, -0.00024722659145481884, -0.000548786367289722, -0.0013916004681959748, -0.00037223706021904945, -0.00021340711100492626, -0.0011884592240676284, -8.254599379142746e-05, 7.921494398033246e-05, -0.0003397847176529467, 0.0006385985761880875, -0.0011482895351946354, 0.0005302699282765388, -0.0034146010875701904, 0.0022135498002171516, -6.2827525653119665e-06, -0.0005223024054430425, 0.000613158626947552, -0.00017130057676695287, -0.0002361031947657466, -2.970270907098893e-05, 0.0005951933562755585, -0.0011547975009307265, 0.0007221261039376259, -0.00045845110435038805, 0.0005792325246147811, -0.0008639489533379674, -0.0011414571199566126, -0.00012732847244478762, -0.0010861876653507352, -0.0006228938582353294, 0.0007742426241748035, 0.0007061089854687452, -0.0005870435852557421, -0.0002600880106911063, 0.00039481345447711647, -0.0008043360430747271, -0.0017217709682881832, -0.0010255920933559537, 0.001054543536156416, 0.0016852064291015267, 0.0003782372223213315, -0.0012988963862881064, 0.00034363323356956244, 0.00039718832704238594, 0.00044024555245414376, 0.0017408138373866677, 0.0010684543522074819, 0.0012773597845807672, -0.0006811556522734463, -0.0002884872956201434, 4.508997517405078e-05, 0.0008383808308281004, -5.409017376223346e-06, 0.0007567059365101159, 0.001059238682501018, 0.0004206834710203111, 0.0018892688676714897, -0.00019089059787802398, 0.0007208955357782543, -0.0007891550776548684, 4.8269892431562766e-05, 0.0008830084116198123, 0.0016548201674595475, 0.00028901748009957373, 0.0013547840062528849, 0.001823977567255497, 2.8471185942180455e-05, 0.0013120726216584444, 0.0015307602006942034, -0.0001652388455113396, 0.0007940577925182879, -0.001803169958293438, -0.0019294830271974206, -0.001188201131299138, 0.0003254646435379982, 0.000706892809830606, -0.0017856369959190488, -0.00022335165704134852, -0.00015001028077676892, -0.0012551168911159039, -0.0003254949115216732, 0.0010194277856498957, 7.298276159417583e-06, 0.0009961372707039118, 0.0008305241353809834, 0.0005843574763275683, -0.00010900437337113544, 0.00033063808223232627, -0.0010286992182955146, 0.0012616808526217937, 0.0018277423223480582, -0.0012290215818211436, -0.0011316575109958649, -2.5952234864234924e-05, -0.0012960756430402398, 0.0005682414048351347, -0.0012508440995588899, 0.0023362680803984404, 0.0019568921998143196, 0.0007127273129299283, -0.0004879020852968097, 0.00121518992818892, -0.00040573993464931846, -0.0009357940871268511, -0.001976340776309371, -0.0008780417847447097, 0.001548964879475534, 0.001473878277465701, 0.0006073119002394378, -2.34095350606367e-05, -0.0014817900955677032, -0.0006081677274778485, -0.000508057652041316, -0.00018386065494269133, -0.0009356178343296051, 0.0007116814376786351, -0.0007089414284564555, -0.0005953912041150033, 0.0005972750950604677, 0.000450065010227263, 0.0011047415900975466, -0.0007846498629078269, 0.0008645503548905253, -0.001706593669950962, -0.00027082115411758423, -0.0010793342953547835, -0.00017642918101046234, 0.0013198298402130604, -0.0022939546033740044, 0.0013138456270098686, 0.0012299037771299481, -0.000462179392343387, -0.0008617010316811502, 0.00028052597190253437, -0.0019141718512400985, 0.0013384184567257762, 0.0018456365214660764, -0.0008038567029871047, -0.0005901079275645316, 0.0006009415374137461, -0.0008267751545645297, -0.0007011428824625909, 0.0007415373693220317, -4.3406191252870485e-05, 0.000671104178763926, 0.0014745082007721066, -0.0008926685550250113, 0.0002059335820376873, -0.00118863710667938, 0.0009988672100007534, -0.0001370135141769424, 0.00018217963224742562, 0.0014103081775829196, -0.001505342312157154, 0.00034979009069502354, -0.0011276272125542164, 0.000491933838929981, -0.0018929721554741263, -0.00048122057341970503, -0.0004585909773595631, 0.0012040241854265332, 0.0003938722366001457, -0.0012208984699100256, -0.0007114760810509324, 0.0010712266666814685, -0.00034583755768835545, 0.00018404850561637431, -0.0005092481733299792, 0.0008676202269271016, 0.0010643716668710113, 0.0022201836109161377, -0.0006752159679308534, -0.0006693100440315902, 0.0001429334661224857, -0.0004022858920507133, -0.001598449656739831, 0.0009046782506629825, 0.0010954015888273716, -0.0014716030564159155, -0.0004925552639178932, 0.0011505857110023499, -0.002196231624111533, 0.00012997526209801435, 0.0001786517386790365, -0.0006230910657905042, 0.0003231626469641924, 0.0004455489106476307, -0.001458592596463859, 0.001028104918077588, -0.00022513548901770264, 0.001500030979514122, -0.001419078791514039, -0.0004991504829376936, -0.0007944359094835818, -0.0006085645873099566, 0.00218698987737298, -0.0004457294417079538, -2.4972157916636206e-05, -0.0001507538981968537, -0.0005284356302581728, -0.0006613674340769649, 0.0006666103727184236, -0.0007713857339695096, 0.0003674376639537513, 0.0006312142941169441, -0.0002501382550690323, 0.00028438985464163125, -0.0008042813860811293, -0.00016355131810996681, -0.0016749872593209147, -0.0008956055389717221, 0.000238428168813698, 0.0013907706597819924, 0.0005084581789560616, 0.0004379848833195865, 0.0007478218758478761, -0.000826395524200052, -0.00029698474099859595, -0.00043197692139074206, -0.0017633249517530203, -0.0013926578685641289, 0.0003635681059677154, 0.0011647403007373214, 0.0003330341714899987, -0.0011288020759820938, 0.0009742048569023609, -0.0005254908464848995, -0.0013604513369500637, 0.000625844462774694, 0.0006893180543556809, 0.0018130348762497306, -0.00040619782521389425, 0.0012990658869966865, -0.00018173000717069954, 0.0005500504048541188, 0.0016409528907388449, 0.00042209323146380484, 0.0008934737415984273, 0.0012853615917265415, 0.0012236012844368815, -0.0005694753490388393, 0.0016066115349531174, -0.00012689923460129648, -0.0006416653632186353, 5.881019387743436e-05, 0.0018760113744065166, -0.0006050362135283649, 0.0008808242855593562, -0.00027369795134291053, 0.0022919264156371355, -0.0014127746690064669, -0.0013474058359861374, -0.0014786503743380308, -0.0002127406478393823, 0.00037971045821905136, -0.0013758878922089934, -0.0020742847118526697, -0.0005276649608276784, 0.0011614002287387848, -0.0019453049171715975, -0.0009252808522433043, 0.0009773888159543276, -0.0002302972716279328, 6.258609209908172e-05, -0.0013971476582810283, 0.0002885485882870853, 0.00030560989398509264, 0.0008530777995474637, 0.0008386404952034354, -0.00011010227171937004, 0.00064699916401878, -0.001272317604161799, -0.0004661218845285475, 0.0007065449608489871, -0.0018558779265731573, -0.0012658281484618783, -0.0006894575199112296, 0.0005375583423301578, 0.0004122369864489883, -0.0010960458312183619, 0.00013681192649528384, 0.0012420082930475473, 0.0012675380567088723, -0.001807474996894598, -0.000457588117569685, 0.0009594555594958365, 0.0011544233420863748, -0.0003894501714967191, -0.00032777836895547807, 0.0008089464972727001, 0.0016098483465611935, -0.0004777419671881944, -0.0014978308463469148, -0.0003835460520349443, 0.0007638183888047934, -0.002403413178399205, -0.0010019871406257153, 0.000386441417504102, -0.0008312963182106614, -0.000292663840809837, 0.00022297099349088967, -2.8577718694577925e-05, -0.0003077110159210861, -0.0006689304718747735, 0.0010575564811006188, 0.001247037318535149, 0.0012086413335055113, 0.0011211222736164927, 0.0010870464611798525, -0.0007613910711370409, -3.3778083889046684e-05, 0.001981001580134034, -0.001336015877313912, -0.0009277607314288616, -0.00014070517499931157, -0.0005788097623735666, 0.0012124241329729557, -0.00042509022750891745, -0.0006285891286097467, -0.0009728075237944722, -0.000238942593568936, -0.0004281105357222259, -0.00024107559875119478, -0.00016030149708967656, -8.215199341066182e-05, 0.0004470044805202633, -0.0004169739841017872, 0.0014601886505261064, -0.0007876601885072887, 0.0009855449898168445, 0.00047642664867453277, 2.42101086769253e-05, -0.0004251523350831121, -0.0003257073403801769, 9.595089795766398e-05, 0.0017361579230055213, 0.0012896563857793808, -0.0010980027727782726, 0.0008471526671200991, -0.00043389928760007024, 3.3987482311204076e-05, 0.0011700702598318458, -0.0003161333443131298, 0.00044105990673415363, 0.0008024533744901419, -0.00036956043913960457, -0.0009946158388629556, 0.00024425756419077516, -0.0001031125575536862, -0.001034581451676786, 0.001514338655397296, 0.0020428618881851435, 0.0009355481015518308, -0.001221438404172659, 0.0013501155190169811, -0.00011441682727308944, -0.0006915002595633268, -0.0010282243601977825, 0.0005577948177233338, 0.001160736894235015, -0.0002247506199637428, 0.0017155224923044443, -0.0014347253600135446, -0.0013554436154663563, -5.446266368380748e-05, 0.001377690234221518, -0.000168086655321531, -0.0004897767794318497, 0.0004403970087878406, 0.0005685186479240656, -0.0008389170398004353, -0.0009270663722418249, -0.00033219545730389655, 0.000768333557061851, -0.0008090445189736784, 0.0006378258694894612, 0.00044920199434272945, 0.001160324551165104, -0.00018525586347095668, 0.0006367202149704099, -0.0003761939879041165, -0.0008947442402131855, -0.00016141781816259027, 0.002012210199609399, -0.0013755675172433257, -0.00020052595937158912, 6.026191476848908e-05, 0.00027003412833437324, 0.001223857863806188, 0.00019808128126896918, 8.5380335804075e-05, 0.0004717133124358952, 0.0001349863305222243, 0.0008327644318342209, 0.0005988263292238116, -2.425789261906175e-06, 0.0011533371871337295, 0.0021484214812517166, -0.00040586531395092607, -0.0002813536557368934, -0.00036015419755131006, -0.001545742736198008, 0.0005248485249467194, -0.0007946674595586956, -0.000332106021232903, -0.0015009436756372452, -0.0016327609773725271, -0.001641952316276729, -6.161374767543748e-05, 0.00228500715456903, 0.00025471573462709785, -0.00014230608940124512, -0.0004997938522137702, -3.925528289983049e-05, -0.0003479369916021824, 0.0011522537097334862, 0.0007447113748639822, 0.000728671089746058, 0.0009169086115434766, -0.001737098558805883, -0.0004436385934241116, 0.0009895816911011934, -0.0003549558750819415, -0.0010570574086159468, -0.000284489564364776, -0.0004379104939289391, 0.0009007928892970085, 0.0005157066625542939, 7.36005385988392e-05, -0.00043670812738128006, 0.0009389755432493985, -1.6742656953283586e-05, -0.0001972947793547064, 6.388638666976476e-06, -0.0006566819502040744, 0.0008075174409896135, 0.0008558369008824229, -0.0010106737026944757, 0.0010027267271652818, 0.00048195268027484417, 0.00024191380362026393, 0.0002399686345597729, -0.0005046717706136405, 6.129029497969896e-05, -0.00121849006973207, 0.00016893312567844987, 9.628714906284586e-05, -0.0008382493979297578, -6.120689067756757e-05, -0.0010857665911316872, 0.00020561354176606983, 0.00023539278481621295, 0.0013835607096552849, 0.0008621496381238103, -2.4932158339652233e-05, -0.001431376556865871, 0.0019802269525825977, 0.0012146682711318135, -0.0002603099273983389, -0.00021098101569805294, -0.001041410956531763, -0.0008037107181735337, -0.001089449506253004, -0.0009898851858451962, -0.001904889359138906, 0.0007401859620586038, 0.0008235988207161427, 0.0014899939997121692, -0.00010204321006312966, -0.001214133226312697, 0.0010107378475368023, 0.0013285254826769233, -0.0013695521047338843, -0.0016377290012314916, -0.0014475920470431447, -0.00030586577486246824, -0.0013461306225508451, 0.0013176167849451303, -0.0005824899999424815, -0.000717564020305872, 0.001317630405537784, -0.0008911647601053119, 0.0006924837944097817, -0.0005762951914221048, -0.00030647506355307996, -0.0020569958724081516, -0.000674407696351409, 0.0007041734643280506, 0.0005926943849772215, 3.0656985472887754e-05, 0.000713917484972626, 0.0012677779886871576, -0.0005432898760773242, 0.0016360227018594742, -0.0007101138471625745, -0.0018005085876211524, -0.0005038612289354205, -0.00038520595990121365, -8.888389857020229e-05, 0.00039472128264606, 0.00024109851801767945, -0.0018261222867295146, -4.781437019119039e-05, -0.0014021467650309205, 0.0004094945325050503, 0.0002945013693533838, 0.0013030859408900142, -0.0013479103799909353, 0.0019076198805123568, 0.0005908364546485245, -0.0013154606567695737, 1.902842086565215e-05, 0.0006740916287526488, 0.0012295530177652836, 0.0006926885107532144, -0.00011228222138015553, 0.0020264573395252228, -6.124739593360573e-05, -0.0017088042804971337, -0.000574132485780865, 0.0010553669417276978, 0.00022877476294524968, -0.00015232438454404473, -0.0006777658127248287, 0.00022405045456252992, -0.00023569021141156554, -0.0012004197342321277, -0.0006825817399658263, 1.886122663563583e-05, 0.0010986966080963612, 0.0009243276435881853, 0.0007693374063819647, 0.0004308061324991286, -0.00021721953817177564, 0.000928495719563216, -0.0006785355508327484, -0.001062864437699318, -0.00030515820253640413, 0.0011326612439006567, -0.0009127494995482266, -0.0006107641384005547, 0.0013263396685943007, 0.00039155231206677854, 0.0006873496458865702, -0.0002301974454894662, 0.0018673035083338618, 0.00040363933658227324, 0.0010758861899375916, -8.759108459344134e-05, -0.0007754300604574382, -0.0002743207151070237], "dtype": "float32"}], "e": {"__tup__": [1.1295179319859017e-05, 1.0041794666904025e-06]}}, {"a": [{"__nd__": [26.041189193725586, 36.296817779541016, -45.03010940551758, 9.489593505859375, -38.632572174072266, -38.39752197265625, -9.579480171203613, -14.938386917114258, -40.237571716308594, -38.41836929321289, 21.413936614990234, 38.27206802368164, 1.2432899475097656, -26.31005096435547, 2.728943347930908, 34.56939697265625, 44.497398376464844, -43.90550994873047, -10.084132194519043, -38.98215866088867, -7.2653961181640625, 1.928713321685791, 30.59389305114746, -33.940521240234375, 0.8783417344093323, -2.413097620010376, 30.253211975097656, 48.12446594238281, -22.428508758544922, -21.997161865234375, -19.634340286254883, -15.065731048583984, -15.489531517028809, -23.326297760009766, -19.5534610748291, 27.701419830322266, -12.760554313659668, -36.04973220825195, -36.385433197021484, -33.613189697265625, 31.488235473632812, -31.872468948364258, 21.465757369995117, -1.1265621185302734, -39.88709259033203, 12.050921440124512, -0.4073123335838318, 15.025599479675293, -6.107726097106934, 42.52543640136719, 44.01416015625, 24.810598373413086, 9.544188499450684, -16.534034729003906, 29.901851654052734, -28.272506713867188, -30.336956024169922, -21.98713493347168, -25.99744415283203, -10.665326118469238, -38.80145263671875, 1.8560163974761963, 33.147911071777344, -27.261398315429688, -15.11651611328125, 48.469764709472656, 6.961275577545166, -1.3385207653045654, -8.659669876098633, -2.205977201461792, 42.531036376953125, -31.35367774963379, 8.798418998718262, 22.518718719482422, 20.058059692382812, -33.70396423339844, 23.076061248779297, 46.276371002197266, -0.8177104592323303, -10.43062973022461, 48.68269729614258, 25.011463165283203, 28.510103225708008, -34.48902130126953, -29.35662269592285, 38.28423309326172, -16.862462997436523, -9.268352508544922, -35.618194580078125, -10.278443336486816, -11.947163581848145, -47.92665100097656, 3.3969902992248535, 43.7337760925293, -11.245981216430664, 6.809741020202637, 14.051436424255371, -25.473129272460938, -49.43233108520508, 31.716140747070312, 23.95513343811035, 13.676003456115723, 42.02470016479492, 26.22283363342285, -19.04542350769043, -25.08489227294922, 7.14447546005249, -49.449893951416016, -26.692663192749023, -41.66907501220703, 22.520263671875, 44.526878356933594, -27.165912628173828, 21.473539352416992, -21.338594436645508, -31.419116973876953, -15.99992847442627, -47.430782318115234, 28.110095977783203, 12.322629928588867, 6.049469947814941, 33.7640495300293, -15.731621742248535, 29.285140991210938, 24.282939910888672, 3.505617380142212, -0.4636129140853882, 5.866497993469238, 16.004070281982422, -44.70055389404297, -26.764921188354492, 35.883583068847656, 21.09249496459961, 42.942867279052734, 45.155296325683594, -43.087860107421875, -38.85032272338867, 2.0174672603607178, 6.835562705993652, -17.2047061920166, 21.44559097290039, 10.011141777038574, -31.854248046875, -31.820697784423828, 28.455379486083984, 44.17091369628906, 22.608095169067383, -20.548891067504883, -49.237117767333984, 7.727339267730713, 6.671777725219727, 21.201492309570312, 1.2800439596176147, 47.962188720703125, -33.3804931640625, 15.800045013427734, -34.46796417236328, -49.03186798095703, 44.834110260009766, -25.56988525390625, -41.835811614990234, 36.14849853515625, -31.601526260375977, 15.806082725524902, 11.325016975402832, -15.61306095123291, -42.20832443237305, 49.35764694213867, -49.87558364868164, 21.811765670776367, -8.788188934326172, 15.516448974609375, 4.590601921081543, 27.97479820251465, -49.426090240478516, -13.591249465942383, -45.253082275390625, -19.91599464416504, 41.449317932128906, 41.69234085083008, -1.6330997943878174, -3.611556053161621, 48.51483917236328, -12.751465797424316, -13.01529312133789, 30.291336059570312, 2.900442123413086, -39.06602478027344, -9.38569164276123, -41.40457534790039, 0.2875939905643463, -47.82416534423828, -8.205183029174805, -38.333351135253906, -49.80586242675781, 20.691545486450195, -31.281187057495117, -33.01092529296875, -42.61064529418945, 26.002315521240234, 4.249267101287842, 25.860746383666992, -29.43353843688965, -44.19595718383789, 15.207833290100098, 15.25925350189209, -7.924170970916748, -45.551307678222656, 33.11031723022461, 16.95383071899414, 12.051478385925293, -13.906241416931152, 12.30797290802002, 6.552528381347656, 30.460725784301758, -12.14088249206543, 19.2927303314209, -48.46424102783203, 29.665517807006836, -27.898794174194336, 27.19327163696289, 1.7914878129959106, 43.133697509765625, 33.920345306396484, 49.981483459472656, 0.6711836457252502, 19.033681869506836, -21.743545532226562, 5.826528549194336, -20.87253761291504, 29.479976654052734, 32.46786117553711, -2.6807210445404053, -45.322784423828125, -36.11343765258789, 47.61624526977539, 45.835575103759766, 37.12175750732422, 33.888484954833984, 48.90623092651367, 2.7422122955322266, 25.018810272216797, -29.816097259521484, 1.446913242340088, -41.30315017700195, -36.41312026977539, -34.359596252441406, -32.083251953125, 49.59088134765625, -38.43865966796875, 24.7575626373291, -6.66488790512085, 33.1873893737793, 3.15352463722229, 10.025102615356445, 25.892333984375, 5.942694664001465, 31.408714294433594, 49.828086853027344, 24.79880714416504, 41.71885299682617, -14.126398086547852, 18.856542587280273, 23.5821533203125, -49.209617614746094, 29.511327743530273, -35.818870544433594, 17.604576110839844, 26.844329833984375, 25.426244735717773, 21.753849029541016, -13.337026596069336, 35.82814025878906, -20.892589569091797, -39.79627227783203, 41.08891677856445, -0.5990208387374878, -16.00327491760254, -46.43267822265625, -42.596317291259766, -13.19027328491211, -30.586307525634766, 28.001365661621094, 5.538007736206055, -39.94309616088867, 4.91826057434082, -24.665447235107422, -13.69826889038086, 24.219484329223633, -26.999536514282227, -30.053770065307617, -5.040529727935791, -5.3927001953125, 48.45664596557617, -27.427339553833008, -36.89509582519531, -2.226696014404297, -29.053163528442383, 26.09187889099121, -15.746367454528809, 34.533241271972656, -40.977989196777344, -21.92711639404297, 5.579828262329102, 38.24006652832031, 41.029029846191406, 27.60622787475586, 13.470590591430664, 39.695125579833984, 2.5610573291778564, -44.844608306884766, 15.689241409301758, 7.942155838012695, -12.620401382446289, 11.814927101135254, -45.157691955566406, -16.40373992919922, -26.42867660522461, 26.814794540405273, -48.27882766723633, -1.8643242120742798, -3.3322579860687256, -29.448400497436523, 35.06974792480469, -48.017574310302734, -10.133496284484863, 10.941924095153809, -40.60735321044922, -17.977861404418945, 15.479266166687012, 4.5988993644714355, -8.27125358581543, 49.64387130737305, -45.93416976928711, -15.054612159729004, 35.778682708740234, -15.416083335876465, 35.82129669189453, 38.10487365722656, 19.399492263793945, 40.60917663574219, 8.56230640411377, -1.2333530187606812, -6.693061351776123, 27.1774845123291, 23.534744262695312, 9.342545509338379, 36.785255432128906, -28.14137077331543, 25.48603630065918, -26.03252601623535, 11.62765884399414, -31.16084098815918, 31.068965911865234, -42.03828048706055, 1.9455816745758057, 49.11068344116211, 39.827091217041016, -33.21454620361328, -22.313533782958984, -0.5036883354187012, -42.891258239746094, 13.811429023742676, -12.486379623413086, 23.69835662841797, 49.07976150512695, 2.3225197792053223, -20.11475944519043, 1.5501506328582764, 29.86713981628418, 43.1312370300293, 9.192569732666016, 42.565345764160156, 38.35977554321289, -46.02702713012695, -39.79446029663086, 4.352749824523926, -45.84352493286133, 12.94388198852539, -24.97229766845703, 34.06031036376953, 46.60528564453125, -22.565324783325195, -14.988974571228027, -44.96910858154297, 31.849878311157227, -29.83783531188965, -35.181156158447266, 31.647558212280273, -48.99424743652344, 48.25039291381836, -4.913469314575195, -44.58790588378906, -48.9925651550293, 0.7945607900619507, 18.02698516845703, 48.53071212768555, -45.970542907714844, -11.297886848449707, 41.902626037597656, 19.191801071166992, -49.533573150634766, 19.47638511657715, -1.333807349205017, 33.490055084228516, -4.389774322509766, -44.579124450683594, -22.446252822875977, 5.116235256195068, -15.914493560791016, -26.29964256286621, -34.12060546875, -13.55020523071289, -9.570555686950684, -12.968802452087402, -37.41358184814453, 32.50127029418945, 45.53225326538086, -17.000370025634766, -10.202798843383789, -14.619513511657715, 28.041465759277344, 43.367401123046875, -19.468069076538086, 33.017704010009766, -26.69415855407715, -15.041776657104492, -21.75369644165039, -42.13285827636719, -18.910018920898438, 6.651642799377441, -49.11135482788086, -24.80052947998047, -40.123687744140625, -22.58169174194336, -44.919551849365234, -13.774883270263672, -45.1067008972168, -31.710105895996094, 5.872303485870361, -19.388736724853516, 17.734067916870117, 24.28873062133789, 0.08848640322685242, -48.005043029785156, -25.110939025878906, 10.968785285949707, 32.40080642700195, 26.72725486755371, -47.447505950927734, 35.390541076660156, 14.746251106262207, 45.910980224609375, 24.43603515625, -25.425722122192383, -17.15428352355957, 33.33137512207031, 49.49196243286133, 29.724559783935547, 39.696041107177734, -10.162714958190918, -23.09977912902832, 29.032337188720703, -48.93724060058594, -15.52552318572998, 32.96934509277344, 9.230830192565918, 40.14033889770508, -39.008113861083984, 43.2591667175293, 4.3973493576049805, 41.14324188232422, -35.3841667175293, -23.36725616455078, 33.58955001831055, -21.09626007080078, -49.275146484375, 11.226901054382324, 35.73431396484375, -48.263397216796875, -21.37497901916504, -20.487743377685547, 8.711461067199707, 1.5361912250518799, -21.999311447143555, 2.729520797729492, -10.511930465698242, 19.763198852539062, -41.93476867675781, 49.33171081542969, -30.10109519958496, 28.147188186645508, 6.786320686340332, 38.84849548339844, 9.514440536499023, -48.573238372802734, 33.95237731933594, -25.716894149780273, -38.247459411621094, 45.96280288696289, -42.29633331298828, -14.68993854522705, -16.321502685546875, -22.392683029174805, -35.84596252441406, 3.305171251296997, 13.835293769836426, 41.573028564453125, 49.010990142822266, -38.73815155029297, -19.18325424194336, -23.164379119873047, 7.3067498207092285, 7.425730228424072, -3.463216781616211, 29.2540225982666, -14.939342498779297, -35.484275817871094, 26.593278884887695, 3.66040301322937, 1.39264976978302, -20.042421340942383, 23.633533477783203, 30.829708099365234, -34.25575256347656, -49.06819534301758, -26.238985061645508, 42.03114700317383, -43.238441467285156, -7.299247741699219, 38.35984420776367, -44.91154479980469, 33.81861877441406, 22.421993255615234, 5.390743255615234, -23.493633270263672, 24.88103485107422, 27.513742446899414, 21.572834014892578, 3.474388360977173, -20.849563598632812, -35.862239837646484, -33.7646369934082, 41.64799118041992, -37.41289138793945, -16.813518524169922, -9.103377342224121, -37.48942947387695, -36.70216369628906, 35.58156967163086, 37.33849334716797, 33.59251022338867, 10.387283325195312, -18.418439865112305, 14.540349960327148, 41.3873291015625, 31.428850173950195, -29.79102325439453, -23.07826805114746, -30.80389404296875, -20.074052810668945, -44.84962844848633, -10.312734603881836, 19.851064682006836, -48.95100021362305, 17.801509857177734, 6.7519307136535645, -9.898427963256836, 0.8554450273513794, -23.031190872192383, -29.375410079956055, 18.31941032409668, 25.81133460998535, 8.661238670349121, -5.387890815734863, 4.084662914276123, -29.222522735595703, 47.695621490478516, -36.20697021484375, 10.49024772644043, 47.035064697265625, -5.2264556884765625, -7.426855087280273, 31.33771324157715, -34.68859100341797, -29.643810272216797, 34.43149948120117, -36.64069366455078, 17.067533493041992, 15.739453315734863, 44.60392379760742, -0.9623202085494995, -28.454092025756836, 6.056110382080078, 23.679828643798828, 20.766788482666016, 24.199359893798828, 24.24091148376465, 27.717838287353516, 7.861222267150879, 19.09840202331543, -13.46595287322998, -30.284822463989258, 0.8752070665359497, -20.96535301208496, 24.570829391479492, -39.38759231567383, -19.27000617980957, 1.378138780593872, -26.727083206176758, 40.921653747558594, -34.25062942504883, -17.287845611572266, 42.67242431640625, -2.812441110610962, -33.88277816772461, -27.63182830810547, 21.05195426940918, -0.45316028594970703, -7.605563640594482, 39.79341506958008, 36.061492919921875, 7.681215286254883, 28.366228103637695, -47.39298629760742, 7.11690092086792, -4.954649448394775, -25.732833862304688, -28.964311599731445, 22.37029457092285, -7.389868259429932, -49.41688537597656, -34.566368103027344, 39.98854064941406, 26.969335556030273, -25.71483612060547, -44.83198165893555, 7.9279680252075195, -0.18411260843276978, -13.113781929016113, -22.037160873413086, -37.64054489135742, 16.063133239746094, -31.196819305419922, 31.352903366088867, -34.27643966674805, 10.457635879516602, -44.0201416015625, 41.90191650390625, 28.729122161865234, 27.767045974731445, 11.028505325317383, -27.48442268371582, 34.7506103515625, -8.851727485656738, -33.877288818359375, -1.0936249494552612, -39.56277847290039, -5.057102203369141, 10.713316917419434, 45.86213302612305, 0.9235861897468567, -40.99892807006836, -34.09062194824219, -49.29510498046875, -19.11200714111328, -2.5987892150878906, -14.878437995910645, 23.804210662841797, -35.76319122314453, 1.160636067390442, 30.96527862548828, -5.842428207397461, -0.2088870108127594, 45.53963088989258, -4.412664890289307, 0.7634139657020569, 49.29099655151367, -9.22777271270752, -12.341978073120117, -12.505581855773926, -12.111092567443848, 21.497112274169922, -43.88777160644531, -15.289777755737305, -33.591270446777344, -46.74928665161133, 49.98886489868164, 46.74585723876953, -28.114641189575195, 3.104656934738159, 27.16481590270996, 14.22672176361084, -20.085058212280273, 48.329444885253906, -15.506867408752441, -6.37196683883667, -11.235041618347168, 28.225727081298828, 20.416292190551758, 6.369903087615967, -25.257726669311523, 7.630849838256836, 46.62445831298828, 30.208480834960938, 45.16497039794922, -36.41982650756836, 46.316497802734375, -49.610958099365234, -41.74974060058594, -28.750167846679688, 29.39636993408203, 49.07789993286133, 38.78900146484375, 11.929543495178223, 11.685774803161621, -26.754932403564453, 19.617961883544922, 24.657184600830078, 2.7305517196655273, -10.565696716308594, -14.929813385009766, -29.884525299072266, 20.583253860473633, 8.845913887023926, 49.07847213745117, -19.26382064819336, -13.384477615356445, 15.753349304199219, -43.35852813720703, 13.3272123336792, 45.13861083984375, 45.78217697143555, -32.042633056640625, -27.362422943115234, 15.709529876708984, 22.27501106262207, 43.58220291137695, 40.46493911743164, 17.019248962402344, 8.124765396118164, 37.00370407104492, 48.35966110229492, 9.379674911499023, 3.063201427459717, 34.0987663269043, -32.84151077270508, 9.695322036743164, -30.35030746459961, 21.4196834564209, -29.54810333251953, 2.6690971851348877, 49.72498321533203, 19.386728286743164, -38.32306671142578, -31.031160354614258, 49.23222351074219, 40.38963317871094, 10.750802040100098, -13.976536750793457, 15.804447174072266, -41.03430938720703, 13.65151596069336, 8.726694107055664, 7.8095855712890625, 11.46651554107666, -42.04705810546875, -14.558597564697266, -28.801753997802734, -28.73841667175293, -11.28843879699707, -9.232460021972656, 10.324475288391113, 3.6529366970062256, 29.082536697387695, -15.676979064941406, -7.816734790802002, 41.042118072509766, 0.04863952100276947, 14.40714168548584, 2.4646193981170654, 14.093031883239746, -37.85567855834961, -5.752804279327393, -4.228836536407471, -15.065632820129395, -28.616769790649414, 25.501649856567383, 48.02119064331055, -9.310044288635254, -34.9476432800293, 17.498592376708984, 49.14186096191406, 49.95551300048828, 14.160305976867676, 18.622461318969727, 10.795732498168945, 3.868433713912964, 46.18528366088867, 19.96851921081543, -36.0343132019043, -27.477296829223633, 47.29133987426758, 5.5372490882873535, 49.32064437866211, 1.2800869941711426, 40.709774017333984, -32.11717987060547, -27.344724655151367, -45.87022399902344, 40.55034637451172, -48.517337799072266, -30.313127517700195, 2.1956663131713867, 42.07267761230469, 47.75565719604492, -39.765567779541016, 28.822933197021484, 9.332256317138672, 44.586421966552734, -3.3064980506896973, -3.838697671890259, 8.386981010437012, 47.716854095458984, 4.452940464019775, 13.937582969665527, -12.016538619995117, 0.3303323984146118, -22.968454360961914, -3.4227805137634277, -8.693272590637207, 42.011253356933594, 16.07959747314453, 28.287639617919922, -44.065834045410156, 14.040034294128418, 46.33939743041992, -34.9767951965332, -35.38738250732422, 39.10258865356445, 44.87941360473633, 24.915882110595703, 22.737730026245117, 17.683727264404297, 46.74702835083008, -42.45429992675781, -26.06296157836914, 23.598407745361328, -44.04508972167969, -20.378936767578125, 19.740211486816406, 32.898475646972656, 48.395240783691406, -1.8326263427734375, -39.29990005493164, 44.74115753173828, 43.76799011230469, 38.15809631347656, -6.733500957489014, 34.94988250732422, 11.611346244812012, 2.453209400177002, -19.05298614501953, -31.195331573486328, 47.30601119995117, 30.041309356689453, -13.265250205993652, 42.70375061035156, -15.217933654785156, 15.50741195678711, 24.584827423095703, -42.59341049194336, 48.79438018798828, 40.14872741699219, -16.121898651123047, -34.670257568359375, 28.23193359375, 21.3455810546875, 6.6399102210998535, -31.377811431884766, -16.361486434936523, -27.868749618530273, 21.02809715270996, 32.89141845703125, -37.93523025512695, -20.322298049926758, -7.400479316711426, -33.17778396606445, 18.586902618408203, -12.419774055480957, 33.16819763183594, 41.73957061767578, -4.362440586090088, -45.8897590637207, 47.52607727050781, -16.276277542114258, 49.69099044799805, 24.643766403198242, 19.945110321044922, 41.60936737060547, 1.1151241064071655, 21.110206604003906, 19.858484268188477, -21.10154914855957, 47.48177719116211, -44.389957427978516, -27.172718048095703, -0.6472135782241821, 17.275283813476562, -4.374042987823486, 30.569351196289062, 15.061880111694336, -44.187618255615234, -45.57463836669922, 6.565759181976318, 20.80643081665039, 33.28097152709961, -29.755207061767578, -33.22956085205078, -11.321316719055176, 2.1332294940948486, -0.32870522141456604, -2.5036234855651855, 33.3349494934082, 32.57575607299805, -46.95769119262695, -49.50205993652344, -1.7715179920196533, 27.238868713378906, -39.83863830566406, 13.383660316467285, 19.294696807861328, 19.329809188842773, 31.624605178833008, -33.86906051635742, 33.36988067626953, 20.35670280456543, 40.12311935424805, -31.70304298400879, 11.231600761413574, -12.157702445983887, -7.106581687927246, 48.75727844238281, -29.10986328125, 47.44775390625, -35.79973602294922, -35.1357307434082, 8.686976432800293, -6.386848449707031, 34.478538513183594, 3.8661093711853027, 41.406349182128906, -3.5052833557128906, 36.083072662353516, 34.086708068847656, -11.995278358459473, 36.57762145996094, 19.433305740356445, -48.9445686340332, 0.4722994267940521, -48.609832763671875, -44.31856918334961, -16.856700897216797, -32.504032135009766, 5.840359210968018, 39.88924789428711, -36.16872024536133, 3.7307088375091553, -2.6349563598632812, -43.03993225097656, 15.942479133605957, 7.125296115875244, 21.183250427246094, -21.086698532104492, -20.449575424194336, -4.2593584060668945, -0.08219911903142929, 16.265932083129883, 10.635536193847656, -19.092315673828125, 18.04796600341797, -1.9708820581436157, -42.85419845581055, 7.704026699066162, 5.877016067504883, 3.5305142402648926, -27.0174617767334, -47.709373474121094, -39.677608489990234, -23.378705978393555, 43.64354705810547, 26.34530258178711, -34.7515754699707, 27.256973266601562, -35.117984771728516, -30.544727325439453, -26.05243492126465, -2.172166347503662, -10.286442756652832, -38.67498016357422, 16.287761688232422, -36.43408966064453, -28.696224212646484, -17.522607803344727, -23.547319412231445, 45.07549285888672, -25.616230010986328, -16.919368743896484, -37.755733489990234, 44.40589904785156, -33.75423812866211, 13.39542007446289, 22.878538131713867, -42.109737396240234, 18.822509765625, 4.169778823852539, 22.708900451660156, 22.623933792114258, -3.228311777114868, 32.082881927490234, -37.83388900756836, 33.803897857666016, 15.571690559387207, 0.9059528708457947, -34.45066833496094, 1.7714123725891113, -27.558956146240234, 30.809396743774414, 0.8069872260093689, 18.77875328063965, 44.141441345214844, 41.66843032836914, 14.605405807495117, 31.76097297668457, -1.9035195112228394, 2.5339863300323486, 43.90848159790039, -46.995208740234375, -12.856975555419922, -6.239867687225342, -35.46414566040039, -26.842897415161133, 13.014203071594238, 41.802734375, 2.382016658782959, -14.2039213180542, -46.60186004638672, 45.77943420410156, 46.32483673095703, -47.996315002441406, -2.414095163345337, 41.24613571166992, 48.8864631652832, 28.676172256469727, -43.661293029785156, 3.3343281745910645, 14.937649726867676, -32.966766357421875, -48.21690368652344, 23.038482666015625, -39.199954986572266, -6.667259216308594, 21.995161056518555, 41.30682373046875, 13.593695640563965, -21.122074127197266, 1.9215439558029175, -4.275949954986572, -47.45884704589844, 41.291419982910156, 14.40431022644043, -30.291135787963867, -35.26433181762695, 11.899487495422363, -35.47745895385742, 42.470664978027344, -48.51455307006836, 41.44784927368164, 48.83007049560547, 28.925880432128906, -49.96806716918945, -42.16217803955078, -7.1420392990112305, 24.38544464111328, -27.610126495361328, 26.59724998474121, 3.279758930206299, 19.63582992553711, 36.249961853027344, 28.601747512817383, 28.579845428466797, -26.21942710876465, 16.069272994995117, 44.83823776245117, -42.61077117919922, -42.03934097290039, 18.744882583618164, 32.74259567260742, -32.116939544677734, 48.257511138916016, 48.344268798828125, -37.20642852783203, -22.453170776367188, -24.07744789123535, -5.052842617034912, -13.870924949645996, 30.454872131347656, 14.926222801208496, -38.71249008178711, 31.672452926635742, -24.911876678466797, 2.1211307048797607, 25.776180267333984, 27.15965461730957, -20.201534271240234, -30.907041549682617, 11.468109130859375, -39.221622467041016, -3.6349098682403564, -37.4683837890625, -45.86875534057617, 2.1400387287139893, 18.246435165405273, 39.93474197387695, 4.028075218200684, -49.965274810791016, 47.914268493652344, 36.94369125366211, 25.501354217529297, -4.178956985473633, -27.687089920043945, -41.60529708862305, 30.819290161132812, -22.499956130981445, 20.07677459716797, 40.002471923828125, 1.5251142978668213, -30.70648193359375, 31.13745880126953, -19.881826400756836, 44.72032928466797, -27.825580596923828, -1.197644591331482, -28.68729019165039, 10.449050903320312, 25.237001419067383, 16.31215476989746, -9.867069244384766, 14.536764144897461, 30.740657806396484, -28.035228729248047, 34.963279724121094, -25.89200782775879, 44.09769058227539, 22.623180389404297, -44.78154754638672, -0.9955912828445435, 11.087196350097656, 48.62898254394531, -26.476930618286133, -48.169681549072266, 40.89381408691406, -36.961997985839844, -1.800361156463623, -31.95145034790039, 26.92732810974121, 35.77090835571289, 11.045609474182129, -23.52562713623047, -27.972301483154297, 25.583789825439453, -28.780010223388672, 48.35836410522461, 31.051565170288086, -10.748932838439941, 6.49574089050293, -42.82229995727539, 14.903824806213379, 20.10152816772461, 11.771686553955078, 13.736345291137695, -48.392616271972656, -3.5851476192474365, -35.67422103881836, -13.989418029785156, -28.252328872680664, -11.484975814819336, -0.7752748727798462, -37.90795135498047, 23.11570167541504, 14.198421478271484, 44.0606803894043, 20.67258644104004, -35.787418365478516, -46.2775764465332, -34.153133392333984, -15.060932159423828, 12.937079429626465, 18.600215911865234, 46.36405563354492, -33.2144889831543, -35.843170166015625, -47.52898025512695, -16.913543701171875, 6.443243503570557, -37.966922760009766, 33.071441650390625, 35.794090270996094, 49.71124267578125, 15.16274356842041, -4.470644950866699, -19.10556983947754, 16.531822204589844, -30.49504280090332, -33.53516387939453, -20.635562896728516, 5.529545307159424, -37.615440368652344, -30.094133377075195, 38.05832290649414, -14.869189262390137, -4.355254173278809, -4.800054550170898, -41.78137969970703, -1.1187818050384521, 22.05661964416504, -2.5488133430480957, -31.15496826171875, 9.329728126525879, 40.17374801635742, -34.05473327636719, -45.1064453125, 20.451431274414062, 39.005043029785156, 18.383716583251953, 39.18256759643555, 9.172248840332031, 49.45866775512695, 20.15364646911621, 43.254058837890625, 18.505613327026367, -45.51614761352539, 6.298868656158447, 40.13112258911133, -30.693798065185547, -21.74514389038086, -8.943108558654785, 45.9040412902832, 46.840972900390625, 6.216317176818848, 42.15798568725586, -19.761953353881836, 6.552279472351074, 2.8199832439422607, -21.33843421936035, 27.208267211914062, -17.902692794799805, 43.545677185058594, 13.512948036193848, -45.01852798461914, -25.117938995361328, -1.7776780128479004, -15.19322681427002, 13.163747787475586, 17.83598518371582, 24.037900924682617, 25.214611053466797, -7.996676445007324, 34.79048538208008, -28.445613861083984, 21.698040008544922, -40.035743713378906, 31.89392852783203, -26.622482299804688, -19.069904327392578, 4.843564987182617, -26.297210693359375, 6.937758922576904, -33.83991622924805, -47.46529006958008, 16.9530086517334, -28.517330169677734, -49.11069107055664, 12.970664024353027, -0.09489350020885468, -23.68134117126465, 20.55471420288086, -25.02231788635254, -43.02187728881836, 15.82832145690918, -20.834884643554688, -39.659515380859375, -33.88724136352539, -10.244932174682617, -38.162925720214844, -5.760383605957031, 33.447994232177734, -13.824837684631348, -20.772750854492188, -44.92401123046875, 7.959610462188721, -25.44675064086914, 33.93655014038086, 45.90323257446289, -48.126686096191406, 21.680896759033203, 8.950392723083496, -29.408584594726562, -26.636253356933594, -24.621740341186523, 32.097896575927734, -6.1889753341674805, -41.166385650634766, 13.269163131713867, -38.43051528930664, 41.07600784301758, -6.817498207092285, -45.54690933227539, -13.833617210388184, 26.517127990722656, 22.54795265197754, 7.102536678314209, 7.999767780303955, -22.67856788635254, -46.832000732421875, -28.47475242614746, 48.11384582519531, 32.755950927734375, 33.21263122558594, -5.023942947387695, 10.907733917236328, -15.95214557647705, 30.463882446289062, -14.69849681854248, -34.9110221862793, -23.870981216430664, 21.351398468017578, -11.407478332519531, 43.67456817626953, 15.472772598266602, 31.81267738342285, 32.79942321777344, 25.50825309753418, 28.814414978027344, -27.9759578704834, -30.489152908325195, -20.132781982421875, -46.898414611816406, -45.887046813964844, 0.2983354926109314, -7.223414421081543, 34.114627838134766, -11.028887748718262, 0.6782728433609009, 34.76192092895508, -35.49725341796875, 17.051103591918945, 0.2402624785900116, -14.512360572814941, 11.419318199157715, 27.31613540649414, -33.912445068359375, -8.044820785522461, -23.442325592041016, -48.55615997314453, 12.524008750915527, -34.47492980957031, -30.697904586791992, -43.60263442993164, -49.421077728271484, 17.562162399291992, 22.47696304321289, -41.63905334472656, -25.67929458618164, 39.62157440185547, 12.00547981262207, -2.1021831035614014, -34.52263259887695, 8.31906795501709, 40.874900817871094, 17.967544555664062, 15.89887523651123, -38.22477722167969, -0.19854192435741425, -14.928038597106934, 16.257770538330078, 36.242881774902344, 24.32642364501953, 19.424392700195312, 26.717435836791992, -13.051583290100098, -6.353734493255615, 8.161397933959961, -33.05870819091797, 12.043375015258789, 25.086145401000977, -33.57123565673828, 27.072067260742188, -38.373477935791016, 9.9605073928833, 46.14952087402344, 34.964412689208984, -24.861528396606445, -42.53139877319336, 44.72233963012695, 40.843597412109375, 8.331694602966309, -48.145233154296875, 2.568767547607422, 27.844491958618164, 12.399807929992676, 20.166311264038086, -6.376718044281006, -7.317593097686768, 25.2552547454834, 3.8739583492279053, 1.0322269201278687, -24.90692138671875, 47.525272369384766, -46.54276657104492, 45.72689437866211, -32.630149841308594, -39.382015228271484, -44.1458740234375, 9.45521068572998, -39.37664794921875, 12.426399230957031, -28.252235412597656, -25.775842666625977, 30.575986862182617, -19.41141128540039, -30.531085968017578, 41.76130676269531, -0.6241550445556641, 6.947412490844727, 33.711585998535156, -35.26646041870117, 9.998741149902344, -18.50743865966797, -18.70535659790039, 35.048980712890625, -8.105292320251465, 13.894980430603027, 40.09640121459961, -45.602054595947266, 6.765224456787109, -31.749235153198242, 13.053125381469727, -44.88777160644531, 13.205211639404297, -20.83220100402832, 4.186320781707764, 3.3630852699279785, -18.04148292541504, 40.378082275390625, 13.577887535095215, 32.44053268432617, 16.508230209350586, -24.539897918701172, -48.70222854614258, 35.32337188720703, -49.747406005859375, -0.48497286438941956, 25.961406707763672, 19.705337524414062, 46.52522659301758, -2.504329204559326, 22.408527374267578, 1.7929314374923706, 42.977821350097656, -16.122568130493164, 30.23517417907715, -17.60405158996582, 47.17213439941406, -40.32911682128906, 28.697185516357422, -8.7676420211792, 9.985100746154785, -39.03588104248047, -45.27541732788086, 22.4075984954834, -31.608623504638672, 14.323960304260254, 5.744530200958252, -22.43259620666504, -33.971187591552734, 11.744140625, 20.159582138061523, -5.438151836395264, 22.915712356567383, 17.30928611755371, 29.65708351135254, 6.50264835357666, 46.505340576171875, -47.743534088134766, 48.76702880859375, -39.170955657958984, -17.69296646118164, 9.075860023498535, -13.325767517089844, -29.385217666625977, 3.898268222808838, 1.2065486907958984, 49.80349349975586, -25.68060874938965, 4.249899387359619, 25.865753173828125, 34.83656692504883, 25.398418426513672, -0.10994936525821686, -11.163866996765137, -10.635588645935059, -3.3400232791900635, 44.39336395263672, 47.74436950683594, -3.3091249465942383, 8.34019660949707, -19.49677276611328, -48.95376205444336, -15.371267318725586, 37.31432342529297, -41.366729736328125, 8.0332612991333, -40.360958099365234, 1.8261274099349976, 6.954159259796143, 40.7257194519043, 19.22627067565918, 36.16071319580078, 17.563142776489258, -19.620838165283203, 6.920187473297119, -38.9445686340332, 11.803725242614746, 40.2966423034668, 37.72372055053711, -46.27553176879883, -49.726070404052734, 46.92671585083008, 21.025022506713867, -48.83037567138672, -43.64482116699219, 9.731249809265137, -24.45444679260254, -13.094874382019043, -1.2725303173065186, 32.89617156982422, -13.487517356872559, 47.5540771484375, 44.92514419555664, -0.6476756930351257, 41.378150939941406, -48.15228271484375, -7.337717533111572, 49.952430725097656, 16.12130355834961, -49.9874382019043, -21.218130111694336, -9.48587417602539, -20.859786987304688, 13.429139137268066, 24.500619888305664, -36.405738830566406, 21.70131492614746, 48.563995361328125, 31.1992244720459, -41.979530334472656, 49.036842346191406, 35.7321662902832, 35.457027435302734, -1.0479965209960938, -9.129392623901367, 43.48096466064453, 1.015409231185913, -26.724767684936523, -8.783475875854492, 43.32878494262695, 21.763832092285156, -36.019283294677734, 36.7988395690918, -21.844348907470703, 2.3996076583862305, -17.69570541381836, -24.495784759521484, 29.007671356201172, -26.336767196655273, 28.444759368896484, 29.491121292114258, -18.212583541870117, -25.581571578979492, 48.88057327270508, 5.394396781921387, 34.73384094238281, 40.26331329345703, -24.763059616088867, 30.025407791137695, 43.09379196166992, -34.7243766784668, -4.547770977020264, 31.71969223022461, -33.40288543701172, 12.449175834655762, 16.56287384033203, 10.167072296142578, 11.376272201538086, 25.876237869262695, 46.08859634399414, -15.939752578735352, -18.568359375, 11.907812118530273, -27.0620059967041, -42.99974822998047, 30.31189727783203, -46.76701354980469, 10.065703392028809, 36.845176696777344, -11.621623039245605, 25.98160171508789, 22.20876693725586, 45.35232162475586, -43.82545471191406, 1.8601808547973633, 1.514159083366394, -47.35502624511719, -48.8216552734375, 35.102638244628906, 44.42021942138672, -20.51711654663086, -19.87267303466797, 22.77461051940918, 23.069875717163086, -29.11857795715332, -33.23252487182617, -33.780513763427734, -2.4216666221618652, 34.10551071166992, 48.874202728271484, 22.843557357788086, 4.35085916519165, 44.584739685058594, -33.4760856628418, -38.63475036621094, 30.443981170654297, -27.618392944335938, 8.487814903259277, 22.044654846191406, 18.91295051574707, 34.595333099365234, -47.50654983520508, -46.253150939941406, -0.713270366191864, 23.241954803466797, -29.183238983154297, 44.203041076660156, 10.227208137512207, -16.14008331298828, 38.28269958496094, 26.16014289855957, -48.50856399536133, 5.269263744354248, -36.184181213378906, -9.9480562210083, -6.083251476287842, -16.064786911010742, 38.760009765625, 9.422091484069824, 17.732702255249023, 43.65098190307617, 15.952178001403809, 27.64134407043457, 22.487377166748047, 0.2601572573184967, 23.030757904052734, -10.052068710327148, 30.07143783569336, 43.33635711669922, -21.758302688598633, -6.9843902587890625, -37.91265869140625, 35.42704772949219, -38.00785446166992, 42.496986389160156, 11.342151641845703, 16.132564544677734, -7.699221134185791, -40.072933197021484, 39.21696090698242, -3.15401029586792, 28.297170639038086, -39.90193557739258, -31.094135284423828, -34.11027145385742, -33.007476806640625, 27.714597702026367, -44.79774475097656, -26.326032638549805, -35.16697311401367, 7.994565963745117, -44.15798568725586, 27.61838722229004, 28.885150909423828, 10.875782012939453, -33.16936492919922, -34.35403823852539, -35.70246124267578, -38.085548400878906, 23.011425018310547, 31.849992752075195, -4.446620464324951, -49.13276290893555, -25.783479690551758, 45.89611053466797, -32.99832534790039, 5.096394062042236, 7.091801643371582, 38.57377624511719, -23.59868621826172, 43.90413284301758, 18.432373046875, 25.21412467956543, -37.12111282348633, 34.10710525512695, -48.333099365234375, 27.810344696044922, -0.26209262013435364, -3.602900505065918, 13.490076065063477, -46.468910217285156, -16.598384857177734, -14.873391151428223, -46.885684967041016, 6.07468318939209, -34.55768585205078, -46.65660858154297, -19.90534019470215, -3.8870017528533936, -33.914817810058594, 31.7930965423584, 15.830552101135254, -33.4757080078125, -30.957998275756836, 11.755523681640625, -22.508708953857422, 22.786075592041016, 22.736812591552734, -37.53520965576172, 5.289998531341553, 38.7628288269043, 32.49901580810547, -36.757259368896484, 14.621679306030273, -28.28953742980957, -45.256813049316406, -17.611360549926758, -23.09005355834961, -34.367366790771484, 14.876904487609863, 13.449637413024902, -47.791770935058594, 33.48680114746094, 14.387760162353516, -6.267707347869873, -31.29226303100586, -21.006319046020508, 15.274136543273926, 34.256324768066406, 7.826929569244385, 19.296646118164062, -28.84712791442871, -23.234067916870117, -28.273212432861328, -39.83667755126953, -3.483025550842285, 46.80596923828125, -47.604103088378906, -22.839462280273438, -23.514488220214844, 19.142780303955078, 33.217777252197266, -26.12991714477539, 23.0430850982666, -40.403560638427734, -9.377169609069824, 32.861019134521484, -15.07800006866455, 35.18397521972656, 36.88388442993164, -7.401342391967773, 40.46050262451172, -23.220691680908203, -31.49333953857422, -36.29188537597656, -46.30136489868164, 22.038837432861328, -22.829105377197266, 25.809301376342773, -1.0046077966690063, -8.144498825073242, 39.426513671875, 33.131954193115234, 34.8746337890625, -23.67060089111328, -39.95093536376953, 1.4886246919631958, -42.29097366333008, -44.76296615600586, 41.65437316894531, 48.56226348876953, -43.95943069458008, -45.09518051147461, 11.928680419921875, -30.292095184326172, 36.57695388793945, 3.6396803855895996, -26.602481842041016, -6.582738399505615, 31.18087387084961, 29.48422622680664, -0.40194958448410034, 37.440818786621094, 20.251005172729492, -43.14012145996094, 36.31526565551758, 34.935306549072266, 11.154067993164062, 21.049535751342773, 20.434350967407227, -21.073631286621094, -45.38764953613281, 33.58955001831055, -22.25758934020996, 34.02365493774414, -13.665409088134766, 46.614356994628906, -37.19184875488281, 19.570079803466797, -45.59125518798828, -47.91543960571289, -39.088871002197266, -46.97932434082031, -8.199300765991211, -37.88421630859375, -36.92589569091797, 37.11488342285156, 14.349555969238281, -22.848861694335938, 17.932825088500977, 49.22117614746094, -49.10076904296875, 22.085676193237305, 41.455997467041016, 25.9698429107666, -12.942712783813477, -30.958494186401367, -44.872802734375, 38.16557312011719, -47.60014724731445, -22.271886825561523, 1.5327000617980957, 34.49789047241211, -0.940149188041687, -45.10806655883789, 16.57965850830078, 39.64791488647461, -44.84361267089844, -10.313380241394043, 9.566230773925781, -12.563241958618164, 21.558032989501953, -24.190176010131836, -11.911066055297852, 37.18814468383789, -5.333691120147705, -10.420230865478516, 44.19268798828125, -8.167810440063477, -12.589768409729004, -36.61867904663086, 13.917160034179688, 11.09558391571045, 17.228351593017578, 48.38365173339844, 37.16636657714844, -24.665992736816406, 47.11311340332031, -4.365715503692627, 22.43475341796875, -21.36018943786621, -4.8011474609375, 47.825889587402344, 29.43439483642578, 18.839174270629883, 39.390567779541016, -27.275222778320312, -42.19290542602539, -21.330711364746094, 20.495811462402344, 37.00116729736328, -26.17009925842285, -2.014660358428955, 21.661775588989258, 4.996948719024658, 39.23009490966797, 45.877662658691406, -49.19087600708008, 16.45155906677246, -20.5546817779541, 0.45873236656188965, -2.110989570617676, -36.51011657714844, -44.993125915527344, -48.012115478515625, -11.097794532775879, 27.72409439086914, 25.882587432861328, 34.63885498046875, 2.340841293334961, 17.895111083984375, -32.94517517089844, -4.138746738433838, 12.487726211547852, 4.95354700088501, -34.401145935058594, 36.67158508300781, -1.7259857654571533, 41.22590255737305, 12.935042381286621, 0.3863597810268402, -3.641388177871704, -11.55787467956543, 32.43558120727539, 12.626023292541504, 29.369993209838867, 15.400677680969238, -30.152883529663086, -42.539573669433594, -39.40645217895508, -3.865917205810547, 13.032397270202637, 15.333617210388184, -27.703752517700195, -36.50619125366211, 20.6209774017334, -42.67020034790039, -10.464183807373047, -17.4117374420166, 32.236976623535156, -40.79477310180664, 0.3912426233291626, -18.706491470336914, 5.218107223510742, -20.951847076416016, -27.4782657623291, 21.565820693969727, 32.72652053833008, -38.92526626586914, 38.82575225830078, -1.417791485786438, -47.48072814941406, -20.30594825744629, -41.94863510131836, 14.89659309387207, 5.652193546295166, -2.6032497882843018, 39.43328094482422, 5.041446685791016, 22.10478401184082, -27.713035583496094, -24.314443588256836, -35.50140380859375, 47.56355285644531, 43.31193542480469, -8.8772554397583, 25.916810989379883, 28.354694366455078, -9.618372917175293, -3.8080697059631348, 44.77542495727539, 15.556660652160645, 5.923003196716309, -44.433631896972656, -31.7316951751709, -2.590811252593994, -27.0601749420166, 19.58695411682129, 23.637834548950195, 15.41081428527832, 11.723370552062988, -12.643858909606934, -38.0785026550293, 18.420976638793945, 18.043466567993164, -27.464258193969727, 38.22829055786133, 43.265480041503906, 43.1004753112793, 0.5747292637825012, 19.39441680908203, -25.29613494873047, -4.561722278594971, 11.128140449523926, 15.167669296264648, -44.723663330078125, 6.419227123260498, 24.005842208862305, -35.31807327270508, 29.666101455688477, -13.365126609802246, 48.595176696777344, 29.66983413696289, 13.780951499938965, -0.523747444152832, 19.929384231567383, -5.766751289367676, 16.265653610229492, 21.144548416137695, 15.835580825805664, 0.6103385090827942, 20.551048278808594, 19.393871307373047, -23.20197296142578, 8.562308311462402, 8.425750732421875, -47.41526794433594, 35.396671295166016, -4.132779121398926, 13.869878768920898, -25.441211700439453, -23.029329299926758, 16.7890682220459, 32.84491729736328, -34.65730285644531, 27.46012306213379, 19.753713607788086, -28.706483840942383, 49.938411712646484, -19.95766258239746, 49.807373046875, -23.90031623840332, 47.326499938964844, 47.31633377075195, -21.64569854736328, 29.758485794067383, -7.580129623413086, 44.65446853637695, 21.589763641357422, 5.884263038635254, 12.4606294631958, -24.59610366821289, 37.602848052978516, 2.7771031856536865, 49.1759147644043, 0.6362566947937012, -22.05079460144043, 12.833602905273438, 10.958166122436523, 13.14374828338623, 30.763948440551758, -7.648857116699219, 47.463653564453125, 8.01638412475586, 18.36910629272461, -3.3372135162353516, -14.584498405456543, -35.885929107666016, -34.77787780761719, -20.917373657226562, 2.101162910461426, -30.174083709716797, 16.839935302734375, 17.036550521850586, -44.29122543334961, 28.79187774658203, -21.22007179260254, 47.43059158325195, 49.76284408569336, -48.564571380615234, 17.37307357788086, 19.445846557617188, -39.51995086669922, -48.000518798828125, -7.114593982696533, 10.749537467956543, 15.741765975952148, -9.227581977844238, -37.02099609375, -32.80259704589844, 39.543556213378906, 34.324188232421875, -44.749935150146484, 39.900699615478516, -48.32733154296875, 37.191410064697266, 33.96183395385742, -8.437414169311523, 15.412683486938477, -16.319913864135742, -47.97502517700195, 7.810403823852539, 35.06055450439453, -17.61227798461914, 17.85755729675293, 42.53301239013672, 37.915618896484375, 30.950883865356445, -22.272897720336914, -29.54729461669922, -13.412742614746094, -24.089401245117188, -6.017456531524658, 16.8398380279541, 13.727038383483887, 49.07425308227539, -46.75342559814453, 11.063688278198242, -42.04502487182617, 14.176949501037598, 46.10151290893555, -41.456058502197266, 19.93184471130371, -35.31416320800781, 46.476104736328125, -46.10609817504883, 7.405030727386475, 49.7934684753418, 38.77973556518555, -44.711795806884766, 0.9680488705635071, 43.52164840698242, 15.998284339904785, -3.883359670639038, 12.16980266571045, -11.814929008483887, 2.273369550704956, 46.9649658203125, 47.97713851928711, -20.741737365722656, -8.8071870803833, 5.7275848388671875, 49.17518997192383, -29.32547950744629, 41.789527893066406, 44.937347412109375, 37.32728958129883, 38.092506408691406, -40.163780212402344, -8.573677062988281, -49.774227142333984, -30.95780372619629, -28.77044105529785, -6.4540276527404785, 18.637775421142578, 38.01333236694336, -7.847358226776123, 23.259984970092773, -30.458736419677734, 18.35387420654297, -43.39361572265625, 18.600831985473633, -7.111279010772705, 3.904498815536499, 13.662256240844727, -25.608808517456055, 49.03189468383789, 2.028045654296875, 4.691389560699463, 24.152591705322266, 39.189857482910156, -5.234716892242432, -40.6165771484375, -20.727712631225586, -34.576595306396484, -29.236209869384766, 10.201664924621582, -27.660566329956055, -27.397119522094727, -43.409446716308594, -35.835906982421875, 18.715688705444336, -7.259133815765381, -30.000179290771484, -49.629295349121094, 25.06655502319336, -46.57945251464844, 45.332542419433594, -14.870484352111816, 28.32457160949707, 22.919099807739258, 7.960402488708496, 28.443605422973633, 6.787760257720947, -44.3516731262207, 10.312543869018555, 7.595343589782715, 12.760601997375488, -43.66877746582031, 26.542861938476562, 35.76569366455078, 35.44646072387695, 17.923986434936523, 29.612638473510742, 34.44215393066406, 34.21516418457031, -46.63497543334961, -6.960492134094238, -14.278627395629883, 32.93767547607422, 8.798785209655762, -25.68547821044922, 21.56003761291504, -26.12993049621582, -5.281219005584717, 46.167503356933594, -21.024600982666016, 35.96965026855469, -29.31586456298828, -49.65342330932617, 21.344362258911133, 36.23978805541992, 29.839567184448242, 15.778282165527344, -14.279844284057617, 18.877023696899414, 16.169382095336914, -11.095457077026367, -41.29433822631836, 26.139175415039062, -44.93321990966797, 4.177400588989258, -5.890837669372559, -25.81702423095703, 40.79051208496094, -27.04221534729004, 15.43460750579834, -42.38447189331055, 16.83287811279297, -0.6903188228607178, 27.37748146057129, 42.215843200683594, -27.187782287597656, -11.93032169342041, 40.466552734375, -42.21068572998047, -14.223234176635742, -11.014302253723145, 47.38004684448242, -0.7207869291305542, -49.447444915771484, 27.832595825195312, -46.583351135253906, -45.075828552246094, -12.01612377166748, -47.038047790527344, 44.27573776245117, 34.953792572021484, 35.464385986328125, -37.48971939086914, 48.13618469238281, 24.74603271484375, -22.9422664642334, 38.5338134765625, 21.88179588317871, 31.867563247680664, 46.0544548034668, -15.713459014892578, -44.91781997680664, 33.60031509399414, -47.15113830566406, 18.169931411743164, -43.130332946777344, 0.2348977029323578, 48.35015869140625, -12.607870101928711, -15.690603256225586, -24.644773483276367, -19.707599639892578, 4.574853420257568, -45.6475944519043, 23.540029525756836, 26.927871704101562, -2.482114791870117, 4.5219807624816895, 9.581891059875488, 40.79893493652344, 42.58328628540039, 15.599153518676758, -31.229442596435547, -37.989341735839844, -4.968163967132568, -23.290264129638672, -46.398460388183594, 11.827366828918457, 34.19419860839844, -29.83462905883789, -25.237043380737305, -20.175363540649414, -20.650327682495117, 35.1755256652832, -46.995384216308594, 7.810571193695068, -49.492156982421875, -23.334951400756836, 29.272611618041992, -35.524723052978516, 12.292537689208984, -32.582218170166016, 46.642452239990234, -11.012760162353516, 6.4859466552734375, -43.74557113647461, -18.861549377441406, 36.29556655883789, -46.136878967285156, -14.682049751281738, -43.117530822753906, -20.823558807373047, -35.26266098022461, 28.52338981628418, -9.638839721679688, -39.873802185058594, 11.50074291229248, -20.34627342224121, 48.7349853515625, -32.06077194213867, -17.555641174316406, 45.68890380859375, -3.389049530029297, -37.34700393676758, 17.873062133789062, 37.17831802368164, -5.281276702880859, -5.182147979736328, 29.95616912841797, -28.876646041870117, -43.72504806518555, -7.350770473480225, -37.10883331298828, -7.89035177230835, -48.244876861572266, -24.057382583618164, -15.768135070800781, -42.756046295166016, 31.09058380126953, -46.71869659423828, -32.756412506103516, -37.09273147583008, -30.808181762695312, -16.000036239624023, 47.217613220214844, 18.286325454711914, 14.281471252441406, -2.7700767517089844, 7.62007999420166, -37.06605911254883, -37.42103576660156, -38.24248123168945, 6.279365539550781, 46.97734069824219, 6.676044940948486, -43.47962951660156, 24.007144927978516, -26.24385643005371, -25.426292419433594, 9.122118949890137, 34.14417266845703, 48.84571838378906, -38.95941925048828, -13.157002449035645, -20.5456600189209, -24.429109573364258, 22.28219985961914, 33.40223693847656, 40.71726989746094, -43.72502899169922, -26.25990867614746, 33.451602935791016, -20.442031860351562, -18.370506286621094, -22.722171783447266, -43.2520751953125, -39.87821578979492, -39.70951843261719, -23.032901763916016, -47.420013427734375, -12.792956352233887, -7.016839981079102, -4.521969795227051, 43.59272766113281, -31.051969528198242, 25.186176300048828, -24.887710571289062, -44.9875373840332, -34.381317138671875, -5.292706489562988, 35.55610656738281, -1.5476508140563965, -40.16514587402344, -34.60236358642578, 1.6261571645736694, 23.003013610839844, -43.485713958740234, -29.835418701171875, 9.915165901184082, 11.268464088439941, 29.921401977539062, 36.432594299316406, -8.289737701416016, 16.144508361816406, -2.1614322662353516, -30.943708419799805, -5.3602447509765625, 36.9199104309082, 6.021007061004639, -29.583248138427734, 26.145822525024414, 45.29814147949219, 49.22286605834961, 17.862520217895508, -35.40165710449219, 6.362204074859619, -36.89004135131836, 12.074716567993164, 19.151721954345703, -0.07730880379676819, 3.1580612659454346, 36.762123107910156, -14.83527660369873, 20.409427642822266, -42.63597869873047, 29.579591751098633, -24.41230583190918, -38.237037658691406, -40.81289291381836, -44.191917419433594, 6.396010875701904, 5.169975757598877, -44.473052978515625, 33.67721176147461, -28.34188461303711, 31.837749481201172, -13.63747501373291, -28.83419418334961, 32.12508010864258, 33.37601852416992, -15.157174110412598, -8.511696815490723, -0.16036450862884521, 13.662129402160645, -31.51323699951172, -46.31690216064453, 35.830474853515625, 37.02321243286133, -14.51762866973877, 16.839017868041992, 27.144180297851562, 1.2287571430206299, 33.97046661376953, -16.44373321533203, 32.17363357543945, -46.041446685791016, 21.39632797241211, 34.78093338012695, 0.9398713707923889, -6.939029693603516, 4.223410129547119, -15.313821792602539, 44.21430969238281, -45.79393768310547, -0.7264875769615173, 3.407331943511963, 35.31396484375, -3.3438711166381836, 31.782886505126953, 1.3479431867599487, 30.808412551879883, 39.2274169921875, -2.9378697872161865, -38.34132385253906, 2.01990008354187, 11.87108039855957, -9.516674041748047, -34.4417724609375, 37.84546661376953, 2.6457161903381348, 41.59272766113281, 22.110868453979492, 43.781776428222656, -42.444427490234375, 16.387882232666016, -40.65605545043945, -39.07480239868164, -37.21961212158203, 20.550386428833008, 33.268516540527344, 23.883203506469727, 14.460503578186035, 31.083078384399414, -40.915618896484375, -3.21982479095459, -28.400306701660156, -25.7528133392334, -15.229229927062988, -21.31950569152832, -33.70553970336914, 28.17298698425293, 2.8372035026550293, 2.491030216217041, -44.23908233642578, -38.806034088134766, 17.57392120361328, 39.61187744140625, 44.948062896728516, 34.34012985229492, -30.445629119873047, -35.190887451171875, 46.12080383300781, 21.535062789916992, -46.51763916015625, -46.16305160522461, -4.3784966468811035, -47.71131134033203, -42.769508361816406, -21.8117733001709, 9.640297889709473, -2.7262864112854004, 0.15607520937919617, 1.058205246925354, 24.30946159362793, -6.237404823303223, 30.72293472290039, 3.841219186782837, -12.846024513244629, 48.44358444213867, 14.860275268554688, -36.8454475402832, 41.832115173339844, 11.821837425231934, -31.024763107299805, 43.88350296020508, 2.7356741428375244, 4.626519680023193, 31.303380966186523, 4.639928817749023, 32.23105239868164, 49.05449295043945, 44.277992248535156, -37.620201110839844, 39.892147064208984, -12.657875061035156, -23.218738555908203, -37.770408630371094, 29.6560115814209, -29.8695011138916, 17.09023666381836, 14.294181823730469, -17.26289939880371, 37.28614807128906, 35.3962287902832, -32.12197494506836, -38.51229476928711, 14.799444198608398, -15.14570426940918, -4.397760391235352, 46.235931396484375, -25.665842056274414, -10.753957748413086, 3.117015838623047, -19.555238723754883, 10.837357521057129, -18.57533073425293, 10.798985481262207, 34.359859466552734, 33.51054382324219, -26.37718963623047, -25.466585159301758, -2.2141337394714355, -45.31752014160156, 9.211461067199707, -6.1326117515563965, -13.315540313720703, -27.90367889404297, -16.487361907958984, 4.864725112915039, -23.55670166015625, 21.81983184814453, 39.51283645629883, -16.560789108276367, -31.41208267211914, 27.422992706298828, -44.04407501220703, -18.75115966796875, -17.30402374267578, -32.5229377746582, 14.180390357971191, 28.445537567138672, -26.78286361694336, -15.441337585449219, 5.554996013641357, 44.883174896240234, 36.6216926574707, 18.536828994750977, -7.0760040283203125, 8.646766662597656, -7.422842502593994, -0.9294100999832153, 37.83537292480469, 17.81928825378418, -27.488128662109375, -0.8943520784378052, 34.51423645019531, -34.77237319946289, -38.46820831298828, 32.60132598876953, -8.716410636901855, -2.631467819213867, 32.59456253051758, -22.311323165893555, -22.400510787963867, -1.3092095851898193, 12.194912910461426, 10.671420097351074, 29.932483673095703, -11.981867790222168, 11.482172966003418, 23.12027931213379, -18.92325782775879, 29.741918563842773, -3.6374261379241943, -43.95301818847656, -15.955204963684082, -5.796010971069336, -38.81100082397461, -4.718648910522461, -31.403905868530273, 12.370034217834473, 19.22224998474121, 24.799463272094727, 32.20232009887695, 42.48210525512695, -16.912220001220703, -31.886274337768555, 14.058853149414062, 36.439231872558594, 2.798764705657959, -20.45530891418457, -22.813899993896484, 38.31550216674805, 36.48253631591797, 36.12057113647461, 46.386505126953125, -0.9784755706787109, -8.013851165771484, -31.354284286499023, 34.54325485229492, 4.8663763999938965, -38.71470260620117, 28.33769989013672, -18.08443832397461, 9.982233047485352, 38.884986877441406, -43.671207427978516, -9.787349700927734, 31.90924835205078, -31.083953857421875, -20.56717872619629, -40.4487419128418, -28.706947326660156, 14.63440227508545, 47.65345764160156, 31.449859619140625, -35.253692626953125, -17.320693969726562, -26.475616455078125, -0.1860649436712265, 28.49021339416504, -38.59668731689453, -11.516098022460938, -38.96347427368164, 41.39205551147461, -25.982126235961914, 47.22264862060547, 40.21189498901367, 41.148841857910156, -36.52602767944336, 33.01991653442383, -44.27330017089844, -14.646636962890625, -14.25816822052002, 26.161043167114258, -33.635772705078125, 18.697805404663086, 48.095523834228516, -1.3477662801742554, -16.16847801208496, 21.760177612304688, -13.915197372436523, 18.01871681213379, -9.14030933380127, 0.6303632259368896, 16.633697509765625, 2.8809001445770264, -20.775558471679688, -40.969993591308594, 38.2432746887207, 18.816959381103516, -45.424766540527344, 46.838539123535156, -37.708740234375, 10.044769287109375, 25.20021629333496, 24.216007232666016, 44.567745208740234, 27.008394241333008, 3.075873374938965, 24.140451431274414, 4.934229373931885, -4.397587776184082, -37.00530242919922, 31.409534454345703, 17.763450622558594, -44.2402229309082, -17.780399322509766, 15.322640419006348, -27.489316940307617, -39.56465530395508, 44.74547576904297, 21.417415618896484, -42.99745178222656, 32.170074462890625, 20.201005935668945, -31.03717613220215, 40.07463455200195, -22.457443237304688, -15.930824279785156, 16.642480850219727, -13.079556465148926, 20.957670211791992, 33.81614685058594, 21.724689483642578, -23.141918182373047, 20.13275909423828, -44.028385162353516, 32.333370208740234, -21.077003479003906, -21.035232543945312, 10.12309455871582, -37.68096923828125, 46.80571365356445, -4.285977363586426, -8.983896255493164, -14.604992866516113, -37.10108947753906, 16.761857986450195, 15.579059600830078, 0.39348578453063965, 35.44567108154297, 33.92124938964844, -29.01251983642578, -2.161214828491211, 49.57465362548828, 44.706825256347656, 45.69119644165039, 19.548904418945312, 34.47825622558594, 33.54069137573242, -33.31611633300781, -46.1610107421875, 47.4372673034668, -40.537330627441406, -3.6970505714416504, 0.37986961007118225, -5.106456756591797, 42.670166015625, 24.04996109008789, -30.986858367919922, -28.784086227416992, -37.475433349609375, -22.768312454223633, 11.509166717529297, -16.99190902709961, -12.29148006439209, -38.90711975097656, 19.841840744018555, 27.536827087402344, 28.014188766479492, -39.539573669433594, 9.158376693725586, -21.573156356811523, -0.876498818397522, -20.718656539916992, 45.546173095703125, 38.74519729614258, 8.20415210723877, -6.092830181121826, 43.994361877441406, -31.053142547607422, -8.945290565490723, 23.243711471557617, -48.743186950683594, -21.641904830932617, -9.615680694580078, -27.663875579833984, 18.07657241821289, 41.60532760620117, -44.893497467041016, 20.234209060668945, 19.72126007080078, -2.963909149169922, 39.80659866333008, -27.955522537231445, 9.156515121459961, 33.46478271484375, -38.26189422607422, -27.21599769592285, -15.506176948547363, -28.035308837890625, -14.733158111572266, 20.82931137084961, 4.093308448791504, -29.606998443603516, 21.34884262084961, -43.987770080566406, -23.57366943359375, -19.150230407714844, -14.965714454650879, -7.581450939178467, 49.51292419433594, 8.478495597839355, -10.162055015563965, -20.42083168029785, 32.39080047607422, -39.45086669921875, -7.988327980041504, 31.644546508789062, 16.904558181762695, 27.419775009155273, -0.4395434260368347, 23.020381927490234, 5.106604099273682, -13.993091583251953, -31.540462493896484, -21.428939819335938, 10.815967559814453, -33.252952575683594, -23.163225173950195, -27.852294921875, -5.6541056632995605, 1.131770372390747, 38.313697814941406, -29.309234619140625, -24.529451370239258, 31.883438110351562, -26.754993438720703, 6.305593013763428, -31.97129249572754, -29.616878509521484, -35.07872009277344, 0.6225783228874207, -29.056333541870117, -18.092933654785156, 39.56718444824219, 15.096088409423828, -45.863250732421875, 12.747462272644043, -21.510120391845703, 15.521388053894043, -10.310824394226074, 2.548372745513916, -37.148162841796875, 39.19097137451172, 47.3957633972168, -4.017983436584473, -32.77347946166992, 23.95943260192871, 19.209836959838867, -23.23816680908203, -40.069068908691406, 19.692026138305664, -22.599246978759766, 7.7514166831970215, 40.779850006103516, 38.15228271484375, -31.969541549682617, 45.80507278442383, -49.8636360168457, 49.75856399536133, -45.96049880981445, -49.157264709472656, -41.09418487548828, 36.52475357055664, 10.441558837890625, -15.446172714233398, 16.266189575195312, 20.630937576293945, -18.400440216064453, -12.558159828186035, -11.379363059997559, -49.2011833190918, 11.528094291687012, 48.618019104003906, 7.144328594207764, 21.940776824951172, -12.197500228881836, 10.074695587158203, -10.423564910888672, 0.03718721866607666, -48.42665100097656, -16.08794593811035, -28.96312713623047, 32.719482421875, -19.14959716796875, -47.75959014892578, -11.824886322021484, -47.07638168334961, 9.900364875793457, 14.891005516052246, -31.55181884765625, 9.11596965789795, -37.0957145690918, 39.61226272583008, 37.62672424316406, 14.772626876831055, -2.1505892276763916, -1.5492711067199707, 14.260746002197266, 38.251766204833984, -16.48335075378418, 22.035259246826172, 14.608150482177734, -7.3388495445251465, -42.61933517456055, -11.126741409301758, 4.334781646728516, 45.72911071777344, -6.437202453613281, 37.56795120239258, 41.99154281616211, -9.548100471496582, -35.70355987548828, 49.38771438598633, 33.63494110107422, -11.357574462890625, -35.20203399658203, -12.087186813354492, -3.6887240409851074, 20.952329635620117, 42.195579528808594, -19.693180084228516, -45.145957946777344, 26.08168601989746, -45.44575500488281, -43.40850067138672, -16.809085845947266, -16.68287467956543, 5.00856351852417, -30.9191951751709, 15.78248119354248, 30.711820602416992, 31.207658767700195, 37.172061920166016, -0.8792726397514343, 3.2752811908721924, -30.622604370117188, -18.213966369628906, -17.084136962890625, 41.475067138671875, -11.184739112854004, 12.207220077514648, -1.2370734214782715, 26.864408493041992, -46.4245491027832, -48.3671989440918, 16.934967041015625, 31.233963012695312, 2.2158379554748535, 21.359512329101562, -39.20580291748047, 11.423818588256836, -39.352901458740234, 42.19398880004883, -48.13605499267578, 0.7353138327598572, -11.770345687866211, 11.185154914855957, -6.037685871124268, 6.347611904144287, 18.6063175201416, -15.553206443786621, -33.583309173583984, -4.557769775390625, 23.657705307006836, -13.085046768188477, 21.135427474975586, -41.18520736694336, 9.488300323486328, 7.474180698394775, 30.831899642944336, 9.835846900939941, -46.270442962646484, -17.456218719482422, -47.458290100097656, -49.47981643676758, 46.25239181518555, -42.16754150390625, 43.369693756103516, -34.14833450317383, 29.305150985717773, -16.682844161987305, 25.484434127807617, 48.70863723754883, -40.089935302734375, -4.688127040863037, 19.016048431396484, 31.473804473876953, -32.124656677246094, 26.73862648010254, -43.82802963256836, -32.635128021240234, 22.329626083374023, -0.4216897189617157, 23.476049423217773, 41.86234664916992, 40.373111724853516, -30.81165885925293, 20.78606605529785, -3.5731911659240723, 12.696237564086914, -29.639604568481445, 31.124601364135742, -16.858524322509766, -37.43515396118164, 21.24031639099121, 43.67474365234375, 27.19968032836914, -4.751481533050537, 31.18955421447754, -15.634489059448242, -45.070560455322266, -21.867555618286133, -24.3532772064209, -8.461076736450195, -28.85521125793457, -30.943891525268555, 5.261720657348633, -22.457820892333984, 47.88374710083008, 40.25583267211914, 22.15450096130371, -49.98835754394531, -28.9953670501709, -12.676115989685059, 41.89482116699219, -0.1578189879655838, 37.21302032470703, -40.98332214355469, -37.853389739990234, -49.542484283447266, -40.79606246948242, -31.099647521972656, 38.890628814697266, -25.255334854125977, 25.002506256103516, -47.94104766845703, -8.298113822937012, 47.30436706542969, 33.57845687866211, -21.973209381103516, -25.337005615234375, -49.284202575683594, -29.874773025512695, -1.0289984941482544, 36.80278778076172, -0.7650172710418701, -34.18543243408203, 21.496374130249023, 17.92083168029785, 38.66796875, 13.801795959472656, 38.98637390136719, -31.05552864074707, -14.228668212890625, -20.03497314453125, 21.138181686401367, -2.329927921295166, -38.586490631103516, 37.54368591308594, 6.0996270179748535, 3.4881157875061035, 40.46049499511719, 33.72549819946289, -41.01522445678711, -3.827350616455078, -34.68719482421875, -49.05224609375, -18.439071655273438, 45.46631622314453, 15.299945831298828, 43.17025375366211, -36.79566192626953, -20.31954002380371, 13.48813533782959, -3.6412384510040283, 1.7198379039764404, -25.868507385253906, 38.756134033203125, -8.34338092803955, 26.72191619873047, -43.91840744018555, -3.2981090545654297, 30.616952896118164, 39.81880569458008, -44.31393814086914, 26.80304718017578, 8.00838851928711, -7.239523887634277, -43.61814498901367, 28.798433303833008, 5.146954536437988, 31.47277069091797, -48.3776969909668, -32.05197525024414, -47.2061653137207, 16.176538467407227, -11.5767183303833, 22.287242889404297, 4.846373081207275, 40.927974700927734, -23.69162368774414, 43.872982025146484, 11.476838111877441, -32.69117736816406, 28.786212921142578, 38.57643127441406, 44.89885711669922, -33.51736831665039, -34.0831184387207, 48.434165954589844, 34.71767044067383, -20.80084228515625, -44.49190139770508, 11.603143692016602, 49.072757720947266, 21.141889572143555, -6.759114742279053, -47.52739334106445, 9.28304386138916, -6.343897342681885, 0.7536116242408752, -33.372474670410156, 42.26028060913086, 45.62837219238281, 37.812198638916016, 41.02507781982422, -27.44490623474121, 27.2425594329834, 13.806304931640625, 30.05862808227539, -44.51911926269531, 38.59914016723633, 23.455062866210938, 25.26186180114746, -6.968148708343506, 5.563567161560059, 0.9317869544029236, 7.484455585479736, 49.87989044189453, -30.726428985595703, 33.56583023071289, 25.758073806762695, -45.08354187011719, 9.478381156921387, -31.428611755371094, -13.620429992675781, -38.228851318359375, -31.912139892578125, -22.268339157104492, -26.948368072509766, -11.158987998962402, 19.42386245727539, -30.190649032592773, -6.234409332275391, -15.445754051208496, 34.391815185546875, 15.310190200805664, -11.799884796142578, 2.672257900238037, 38.59716796875, -43.183956146240234, -48.10910415649414, 7.550994873046875, -20.321821212768555, 21.326820373535156, -8.72263240814209, 6.758300304412842, 0.5766358375549316, -5.848903179168701, -44.80604934692383, 38.354618072509766, 46.26900100708008, 19.27692985534668, 47.21684646606445, 28.80006980895996, -5.460340976715088, -3.2631800174713135, 21.532194137573242, -0.7950971126556396, -44.94679260253906, -11.734786987304688, 2.024383306503296, 14.147144317626953, 41.74211120605469, 14.696563720703125, 41.63884353637695, 44.74453353881836, 6.370574951171875, -10.559913635253906, 42.10464859008789, -46.28031921386719, -31.523151397705078, -25.08853530883789, 16.75601577758789, 48.78022766113281, -35.87911605834961, -10.76503849029541, 34.47148513793945, -29.848323822021484, 19.6652774810791, -17.978836059570312, -22.811931610107422, 13.952601432800293, 6.4196929931640625, 27.31070899963379, -30.80400276184082, -18.304306030273438, -27.813875198364258, 1.1168677806854248, 17.145931243896484, -15.480583190917969, -33.06692123413086, 9.029092788696289, 22.81392478942871, 49.90556335449219, 38.34150314331055, 31.69586181640625, -30.40767478942871, -29.653520584106445, 5.5967535972595215, 23.5764102935791, -11.837884902954102, -18.615768432617188, 42.19913864135742, 44.3974609375, -49.70698547363281, -3.5229828357696533, -40.77013397216797, -41.47631072998047, 3.6828324794769287, 48.73206329345703, -41.58468246459961, -10.037784576416016, 9.286234855651855, 23.771028518676758, 27.607431411743164, 15.413890838623047, -27.83995819091797, -41.28815841674805, -27.254322052001953, 1.9049266576766968, -13.184751510620117, 4.065964221954346, 23.739072799682617, -46.589324951171875, -28.7154541015625, -12.748495101928711, 6.289892196655273, -10.865270614624023, -22.16021728515625, 23.135236740112305, -44.6901741027832, -37.379390716552734, -45.76297378540039, -17.564043045043945, -30.58944320678711, 17.17452049255371, 43.58110046386719, -4.2306036949157715, 42.651119232177734, -49.53093719482422, 46.76033020019531, 27.334733963012695, 5.738821029663086, -47.5477180480957, 27.429397583007812, -13.010944366455078, -24.865764617919922, 29.132627487182617, -40.142276763916016, 28.946596145629883, -47.827598571777344, 40.6182746887207, 46.91474914550781, 33.97676467895508, 26.85480499267578, 1.9187935590744019, -39.235103607177734, -2.69698166847229, 15.898463249206543, -9.604582786560059, 8.60516357421875, 28.504915237426758, 13.488096237182617, 44.45426940917969, -5.524565696716309, -14.297829627990723, 41.899600982666016, 1.6550418138504028, 17.13956642150879, 10.660460472106934, 36.35689926147461, 11.537018775939941, 3.6265292167663574, -45.628787994384766, 12.616268157958984, -20.59524917602539, -36.421939849853516, -35.696044921875, 26.949209213256836, -41.577999114990234, 22.9429988861084, 28.10988998413086, 45.767696380615234, 2.4678473472595215, -3.6790523529052734, -35.42266082763672, -3.0705690383911133, 44.64778518676758, 45.58583068847656, -9.228433609008789, -6.6580281257629395, -16.872011184692383, -26.723764419555664, 35.52109146118164, -0.22516056895256042, 28.1591854095459, -12.333271980285645, -10.700374603271484, -46.505470275878906, -2.8725783824920654, -4.860151767730713, 43.21434783935547, 17.38791847229004, -15.665449142456055, 37.29632568359375, -24.565757751464844, -30.053327560424805, 40.8137321472168, 24.992952346801758, -36.758358001708984, 4.270854473114014, 16.266664505004883, -44.00847244262695, -29.040945053100586, 32.352352142333984, -23.555599212646484, -15.508201599121094, 47.46430969238281, 24.721466064453125, -46.69257354736328, -41.202392578125, 47.95831298828125, 3.092412233352661, 12.128096580505371, -47.89308547973633, 32.46896743774414, 39.28543472290039, 44.496070861816406, 24.36551284790039, 11.03857421875, -9.938539505004883, 10.115567207336426, -35.28159713745117, 45.0528564453125, 37.29547882080078, -37.061676025390625, -14.23050594329834, -0.5107948780059814, -9.761287689208984, -38.35309982299805, -40.681819915771484, 23.978084564208984, -47.24652099609375, -27.53681182861328, 7.117398738861084, -9.24881362915039, 32.13612747192383, -48.87660598754883, -22.734188079833984, 41.08268356323242, -0.7499448657035828, -34.9637565612793, 30.54750633239746, 9.430217742919922, 47.666324615478516, -44.31396484375, -28.63842010498047, 27.62246322631836, -34.592071533203125, -39.98096466064453, 39.444541931152344, 21.382497787475586, 27.766284942626953, -18.125083923339844, 40.34916687011719, -27.563871383666992, -40.23276901245117, -39.140647888183594, -29.635467529296875, -0.22355182468891144, -8.362101554870605, -12.367667198181152, -30.22899055480957, 4.917468070983887, -46.034706115722656, 1.8824236392974854, -40.52854919433594, -14.882076263427734, -9.623687744140625, 24.63131332397461, -48.314693450927734, 21.902175903320312, 43.28874969482422, 43.92536544799805, -18.00336265563965, 35.9304084777832, 39.51506805419922, 17.937835693359375, -44.99037170410156, -19.286941528320312, -47.282466888427734, -8.096091270446777, -39.21883773803711, -6.593233108520508, 19.021223068237305, -37.774654388427734, -23.467784881591797, -33.962989807128906, -4.153914928436279, 2.070207357406616, -42.961082458496094, -15.362910270690918, 41.81483459472656, 19.332927703857422, 41.34408187866211, 37.07069778442383, 36.24363708496094, -27.382596969604492, 8.801032066345215, 22.183372497558594, -46.81660842895508, -1.726117491722107, -45.15339279174805, -48.72561264038086, -9.065441131591797, 29.734392166137695, -22.992647171020508, -10.632431030273438, -28.471712112426758, -31.53913116455078, -12.366959571838379, 32.72416687011719, -31.52714729309082, -2.5042245388031006, 0.9888589978218079, -43.31428527832031, 29.740739822387695, -16.17180061340332, 45.02930450439453, 7.632770538330078, -14.564300537109375, 26.930774688720703, -48.788394927978516, -17.736364364624023, 7.110206127166748, -2.8351776599884033, -31.909521102905273, 30.49664306640625, 8.316598892211914, -2.4116978645324707, -39.58756637573242, 19.209810256958008, -26.4799747467041, -48.054927825927734, -42.609710693359375, -28.95391273498535, 23.215551376342773, 27.57852554321289, 38.78114700317383, -9.870067596435547, 24.899078369140625, -35.07072448730469, -1.730329990386963, -30.64060401916504, -9.571242332458496, -40.852813720703125, -5.502538681030273, 8.380024909973145, 44.624393463134766, 2.7967355251312256, -2.767806053161621, 23.4757022857666, -13.415777206420898, 12.531658172607422, 45.1635856628418, -10.8471040725708, 15.7058687210083, 38.02748489379883, -46.48429870605469, -13.437621116638184, -12.285593032836914, 25.0467472076416, -29.256732940673828, -31.13026237487793, 40.9554557800293, -41.12925338745117, 37.703304290771484, 30.097259521484375, -29.312410354614258, 7.816952228546143, 34.37103271484375, -37.28992462158203, -40.60221481323242, 29.05184555053711, -42.9340705871582, -42.057918548583984, -24.92929458618164, 43.83365249633789, -14.038430213928223, -38.64509582519531, 25.91057586669922, 38.47246170043945, -17.55811882019043, 0.44911521673202515, -23.7847843170166, 25.147838592529297, 9.406881332397461, 44.053077697753906, 48.80022048950195, 33.090885162353516, 33.759979248046875, 6.305789470672607, 16.481603622436523, 47.020545959472656, 33.25262451171875, -7.310388565063477, 7.3618292808532715, -20.382400512695312, -7.656103610992432, 1.898433804512024, 19.48292350769043, 25.4645938873291, -42.02004623413086, 6.427644729614258, 16.72187042236328, -35.812557220458984, -10.397512435913086, -12.318584442138672, -21.550186157226562, -30.795520782470703, -49.249778747558594, -22.143239974975586, -29.79621124267578, -11.330309867858887, 37.54379653930664, 21.3994140625, -42.72834396362305, 38.052738189697266, 20.961231231689453, -12.474041938781738, 36.836021423339844, 5.0296101570129395, -44.37406921386719, 0.3164295554161072, -5.120400905609131, -16.133350372314453, 13.89838695526123, -29.2340030670166, -31.81243133544922, -3.2512218952178955, 21.024757385253906, -23.64044761657715, -43.07180404663086, 46.74544906616211, -24.207096099853516, -21.84405517578125, -40.44681930541992, 42.06361389160156, 6.84455680847168, 43.01795959472656, -15.059982299804688, -33.617225646972656, 29.269390106201172, -31.51930046081543, -17.358949661254883, 30.118391036987305, -32.45979309082031, -26.784467697143555, -19.890188217163086, 32.530269622802734, -7.0375261306762695, -37.44255828857422, 13.439627647399902, -25.82834815979004, -4.882323265075684, -49.663665771484375, -47.40399932861328, -35.370826721191406, -35.302772521972656, 29.501893997192383, -0.07420771569013596, -9.367128372192383, -16.12308120727539, 11.974562644958496, -38.94438171386719, -0.5870723128318787, 16.214927673339844, 25.12128448486328, 38.276695251464844, 5.938922882080078, 25.97272491455078, -27.777971267700195, -10.083271980285645, -5.2638092041015625, -25.47772216796875, 32.727970123291016, -33.525413513183594, 18.27199935913086, 26.58009910583496, -7.722853183746338, 1.7333850860595703, 43.80475616455078, 42.227867126464844, -16.733463287353516, -22.407928466796875, 42.97724533081055, -34.30553436279297, 28.421398162841797, 8.08619213104248, -31.1951961517334, 7.775444030761719, 37.628780364990234, 12.842164039611816, 47.33061218261719, -9.600587844848633, 32.396934509277344, 15.630785942077637, -4.743448257446289, 1.2688632011413574, -15.242659568786621, 25.953594207763672, -21.955028533935547, -31.334522247314453, -12.50822639465332, -48.25223159790039, 8.112990379333496, 1.964726448059082, 8.169793128967285, -28.80318260192871, 47.634578704833984, 28.85667610168457, -1.047341227531433, 0.5364575982093811, -6.440973281860352, -38.58935546875, -20.718870162963867, -31.525218963623047, -24.984928131103516, 41.048988342285156, 46.445152282714844, 46.89384078979492, 22.225122451782227, 34.07539367675781, -33.1024055480957, 9.894203186035156, 28.05046272277832, -35.64173126220703, 25.02965545654297, 12.169967651367188, 29.60015106201172, -43.67179489135742, -29.52336311340332, -37.50674057006836, 9.73012638092041, 19.41154670715332, -4.957233905792236, 15.478351593017578, 49.66736602783203, -20.928281784057617, -13.273807525634766, 37.064735412597656, -35.11166763305664, -24.321577072143555, 11.34338665008545, -34.35740661621094, 14.589591979980469, -45.07343673706055, 47.815185546875, -10.920557022094727, -24.61281394958496, 11.837202072143555, -39.17851638793945, -33.8319091796875, -8.844747543334961, -48.23828125, 26.13200569152832, 18.0386905670166, 6.962184906005859, -29.12735939025879, 39.69778823852539, 14.472132682800293, -26.6641788482666, 31.41535758972168, -35.80911636352539, 19.05600357055664, 4.203085422515869, -7.494536399841309, 12.340499877929688, 8.599808692932129, 47.467769622802734, 37.56088638305664, 32.24333953857422, -32.453636169433594, -5.366153717041016, 0.5721942186355591, -46.392581939697266, -14.623598098754883, -8.500905990600586, 4.6953444480896, -31.296689987182617, -16.281713485717773, 12.41947078704834, 16.002517700195312, 35.53669738769531, 0.3957007825374603, 39.80695724487305, -34.714874267578125, 30.45336151123047, -14.772648811340332, 32.45988082885742, -36.905242919921875, 10.659804344177246, 47.99514389038086, 26.223596572875977, 37.05649948120117, -19.75190544128418, 15.359855651855469, -31.921297073364258, -17.118228912353516, 14.602824211120605, 43.0443000793457, -0.6812617182731628, -31.832677841186523, -13.151994705200195, 18.317249298095703, 18.16109275817871, 43.128265380859375, -12.813041687011719, -3.8961386680603027, -49.337379455566406, 4.4494524002075195, 4.877598285675049, 7.972348213195801, 0.15172947943210602, -14.545228004455566, 29.774343490600586, -24.3017635345459, 20.213510513305664, 35.925453186035156, 4.545614719390869, 14.157437324523926, 10.295195579528809, 23.160961151123047, -3.817007303237915, 39.80790710449219, -42.322994232177734, -11.26673412322998, -22.459617614746094, 7.903666019439697, 49.5364990234375, 20.210988998413086, 45.52796173095703, 40.238948822021484, -49.66323471069336, -33.29314041137695, 9.611725807189941, -27.467126846313477, 30.916542053222656, -31.835609436035156, -2.825308084487915, 8.530801773071289, 13.633989334106445, 7.038503170013428, -22.05084800720215, -42.402801513671875, -1.1651028394699097, -44.1329345703125, -30.479642868041992, 3.026201009750366, 3.5409250259399414, 49.88179397583008, 13.912149429321289, 27.808164596557617, -20.915273666381836, 25.741634368896484, 37.9769401550293, 45.23725509643555, -3.2161402702331543, -38.25386428833008, 7.7460761070251465, -37.95100784301758, 43.028263092041016, 17.523193359375, 24.439796447753906, -46.45401382446289, -21.57465171813965, 23.282957077026367, -34.255062103271484, 14.38976001739502, 31.05620574951172, 39.460086822509766, -32.70570373535156, 22.043684005737305, -1.9701764583587646, 13.233658790588379, 12.328093528747559, -25.06537628173828, -31.93324851989746, 48.701263427734375, 43.9888801574707, 41.71417999267578, -8.521418571472168, -40.871917724609375, -21.08875274658203, -33.73705291748047, 24.794824600219727, -2.01340389251709, -2.9608778953552246, 18.56631088256836, 46.63605880737305, -4.750266075134277, 27.83953094482422, 11.346884727478027, -9.07126235961914, 43.97026443481445, 44.75619125366211, 26.23265266418457, 21.064289093017578, 12.88233757019043, -17.855825424194336, 29.881053924560547, 27.017160415649414, -6.715646266937256, 3.530562400817871, -28.87303924560547, -37.747737884521484, 16.83724594116211, -29.889080047607422, 18.69010353088379, -37.963470458984375, 3.4674859046936035, 20.444421768188477, 40.447017669677734, 35.54750061035156, 11.556567192077637, 21.396080017089844, -14.009657859802246, 5.991420269012451, 41.21002960205078, 23.853673934936523, 36.85030746459961, 30.495338439941406, -9.527863502502441, -12.695457458496094, 9.01231575012207, -39.608402252197266, 45.527950286865234, 42.53752136230469, -24.38217544555664, -12.69257926940918, -25.45237159729004, 34.836727142333984, 0.05867423489689827, -33.92051315307617, 38.24725341796875, -1.8047617673873901, 16.947925567626953, 16.586563110351562, -23.77362632751465, -11.193942070007324, 10.860587120056152, -30.866580963134766, -3.81542706489563, -0.8723965287208557, 41.6485710144043, 23.780860900878906, -8.818795204162598, 48.127784729003906, -1.950725793838501, -12.819560050964355, -41.45193862915039, 21.797822952270508, 48.27893829345703, -0.5457054376602173, -26.894590377807617, -12.478899002075195, 46.620567321777344, -30.619518280029297, 40.272865295410156, 43.51901626586914, -37.74893569946289, 0.8435213565826416, -49.26572036743164, -37.55672836303711, 22.617361068725586, -27.038312911987305, 27.303115844726562, -22.9315185546875, 23.788061141967773, 25.121013641357422, -27.859468460083008, -21.845844268798828, 24.787578582763672, -28.481182098388672, -42.18818283081055, 8.035058975219727, -22.976665496826172, -16.132413864135742, -42.1859130859375, -9.710721969604492, 0.14385125041007996, 41.67363357543945, 42.29084014892578, 5.611825942993164, 28.284313201904297, -40.730445861816406, -5.8835978507995605, -29.71880340576172, 30.36590576171875, -18.35072135925293, -32.25965881347656, 3.5803306102752686, -23.533042907714844, -9.47647476196289, -24.990835189819336, 20.312227249145508, 25.302417755126953, 9.89847183227539, -12.883163452148438, -7.8098063468933105, 6.50930118560791, -0.938642680644989, -32.33567810058594, -12.87118148803711, -43.72481918334961, 13.308527946472168, 41.865684509277344, -40.202049255371094, 11.182211875915527, -4.928230285644531, 27.459014892578125, -43.678646087646484, 44.922603607177734, -0.20187848806381226, -6.13888692855835, -18.525941848754883, 49.62114334106445, 7.352343559265137, -13.597431182861328, 29.33881187438965, 15.821755409240723, -6.6209211349487305, 1.8312591314315796, 14.53824520111084, -22.199932098388672, -7.17948579788208, -3.8050107955932617, 45.574275970458984, 36.45025634765625, 31.415584564208984, 49.83656692504883, 36.7650260925293, -36.75393295288086, 46.821712493896484, 21.960622787475586, -49.198829650878906, -24.123218536376953, -20.406095504760742, 44.32910919189453, -5.925816535949707, 7.777868747711182, -22.433284759521484, 5.529951572418213, -4.211075305938721, -32.031883239746094, 48.119964599609375, -10.392584800720215, 31.459529876708984, -24.262250900268555, -6.4802703857421875, -29.49744987487793, -48.10390090942383, -8.206588745117188, -39.17599868774414, -48.960350036621094, -49.88776397705078, -11.578413009643555, -31.694852828979492, -2.741460084915161, -43.919517517089844, 5.7628936767578125, 6.357297420501709, -14.922005653381348, 7.649127006530762, -44.90534973144531, -6.899344444274902, -30.26908302307129, -34.517852783203125, 16.879398345947266, -20.966856002807617, -41.8963508605957, -31.94573974609375, -9.406057357788086, 16.040515899658203, -49.095211029052734, 4.486542224884033, 44.27326202392578, -24.83133316040039, 15.277832984924316, -2.193437099456787, 7.917695045471191, -44.02952194213867, 11.266680717468262, 15.949933052062988, 47.75748062133789, -28.263277053833008, 15.278010368347168, -46.13064193725586, -39.047630310058594, -49.581031799316406, 33.974365234375, 16.220111846923828, -9.15539264678955, -3.9126970767974854, 2.2736637592315674, 39.20138931274414, -48.38502502441406, 8.077300071716309, -49.60323715209961, -9.74202823638916, -46.191226959228516, 10.999059677124023, 31.063215255737305, 46.926597595214844, -39.19707107543945, -20.669919967651367, 25.251787185668945, 10.090509414672852, 6.940476894378662, 29.10807228088379, -9.265413284301758, -13.17244815826416, 33.769378662109375, 34.626346588134766, -38.82179641723633, -12.91002082824707, -27.367412567138672, 3.9039881229400635, 46.49687957763672, -16.285465240478516, -45.267425537109375, -29.40234375, 12.081938743591309, -26.89212989807129, 43.54117202758789, 28.222396850585938, 23.155609130859375, 40.82373046875, -31.18512535095215, 4.629592418670654, -48.37722396850586, -49.9946403503418, -12.451244354248047, 31.085588455200195, -13.064271926879883, -30.552562713623047, 22.565000534057617, -4.69472599029541], "dtype": "float32"}], "e": {"__tup__": [-0.09996795654296875, 851.8065795898438]}}, {"a": [{"__nd__": [-5.907390594482422, 3.621398448944092, -5.907555103302002, -1.356272578239441, -6.039258003234863, 2.3838160037994385, -1.0808848142623901, -5.994492053985596, 2.0378317832946777, 2.855384349822998, -1.4095131158828735, 1.4630920886993408, -2.072145462036133, -3.55543851852417, -4.413309097290039, -3.0079076290130615, 4.3610429763793945, -2.2357163429260254, 4.442216396331787, -10.292583465576172, -3.2417941093444824, -10.572203636169434, -5.424173355102539, 2.039484977722168, -0.9859157204627991, -1.749538779258728, -1.5745978355407715, -3.7123966217041016, -5.389575004577637, 2.185457944869995, -1.5660203695297241, -7.063313007354736, -4.842989444732666, -4.993915557861328, 0.17309993505477905, -0.33854183554649353, -1.2945064306259155, -2.5668692588806152, -4.446605205535889, -4.522527694702148, -2.632152557373047, -0.8056877255439758, -2.342702865600586, -1.7983918190002441, -4.057204246520996, -0.5941488742828369, 0.3014768362045288, -6.448273658752441, -0.36101311445236206, 0.14455586671829224, -7.87786340713501, -7.096076965332031, 0.10643276572227478, -1.1582673788070679, 0.22336778044700623, -0.8905511498451233, -4.060732841491699, -3.9192283153533936, -6.093331813812256, -1.6438366174697876, -2.5287396907806396, 2.0705418586730957, -6.167953014373779, -0.40344494581222534, 1.2881377935409546, -3.512716054916382, -3.150161027908325, -4.631876468658447, -4.711307525634766, -1.4073117971420288, -10.179840087890625, -7.5425591468811035, 4.379261493682861, -4.759119510650635, -4.5724077224731445, -5.156527996063232, 1.4032825231552124, -2.9182565212249756, 1.0629321336746216, 1.4575198888778687, -8.727394104003906, -1.7782056331634521, -3.209141254425049, -4.6107940673828125, -1.4914240837097168, -2.226316213607788, -0.4479813575744629, -8.266963958740234, -1.1252965927124023, 0.40984904766082764, -2.0576038360595703, -5.0726799964904785, -1.8195030689239502, -0.13979008793830872, 2.0632998943328857, -4.969704627990723, -2.475087881088257, -2.733687400817871, -3.545654773712158, -3.477647542953491, -0.2776220440864563, 3.556406259536743, -4.008465766906738, -1.7949777841567993, -2.1408369541168213, -2.764719009399414, -7.751680374145508, -5.468996524810791, -2.555457592010498, 0.0722186490893364, 0.28477925062179565, -3.505791425704956, 4.5283522605896, -3.8476614952087402, 0.4476085305213928, -4.665200233459473, -2.053510904312134, -3.83438777923584, 0.31888437271118164, -0.7020719051361084, -3.124765396118164, 2.1039466857910156, -0.485620379447937, -1.9034924507141113, 2.666074275970459, -3.025117874145508, 3.511284112930298, -5.584197998046875, -1.5396684408187866, 1.4029359817504883, -5.558722496032715, 1.5540579557418823, -7.412148952484131, -0.4528789520263672, -0.1959979385137558, -0.4452531039714813, -2.46716570854187, -1.8669143915176392, -2.6622390747070312, -7.077536106109619, -0.4816298484802246, -0.386254221200943, -2.0219972133636475, -3.4897336959838867, 1.0494792461395264, 0.5788062810897827, 3.6765286922454834, -3.535339593887329, -6.675978660583496, -3.9514548778533936, -2.1694180965423584, -0.6622400879859924, -3.6045188903808594, -2.3104348182678223, 1.6134463548660278, -4.311697006225586, -2.169527530670166, 0.2406674474477768, 0.28101569414138794, -1.1561236381530762, -3.8529481887817383, 0.8599472641944885, -2.9538021087646484, -5.8113789558410645, -2.1061668395996094, -7.325913429260254, 3.0799202919006348, -1.793519377708435, 5.7581400871276855, -4.304437637329102, -6.967984676361084, -2.893474578857422, -0.4945940673351288, 1.4783679246902466, -3.1119918823242188, -3.496854066848755, 0.6493930220603943, -4.313021183013916, -1.5251920223236084, -3.3830363750457764, -0.09452934563159943, -0.23495234549045563, -0.5346250534057617, -2.568513870239258, -8.39600658416748, -1.2184925079345703, -1.6492977142333984, 1.0980756282806396, 0.4892346262931824, -1.6644134521484375, -1.6224467754364014, 2.327878713607788, 0.10220129787921906, -1.878420114517212, 0.313953161239624, -1.9590318202972412, -2.444570302963257, -0.18524512648582458, -1.156198263168335, 3.5589613914489746, -4.897052764892578, -6.502338886260986, 1.0145583152770996, -3.257981777191162, -2.104158878326416, -2.576960325241089, -1.9456652402877808, -6.318775653839111, -3.075655460357666, -1.7457209825515747, -0.9321167469024658, -1.3346914052963257, -0.6018367409706116, -0.3655076026916504, -6.159334182739258, 1.8398343324661255, -8.128704071044922, -0.7338577508926392, -5.280003070831299, -2.745466947555542, -5.466064929962158, -0.45420828461647034, -5.348453521728516, -3.7122395038604736, 0.05997664853930473, -6.549984931945801, -0.40248727798461914, -3.0468087196350098, -3.4847805500030518, -5.444882392883301, 0.8454153537750244, 1.0991588830947876, -0.5810135006904602, -7.204748153686523, -1.3654264211654663, -3.2504258155822754, -3.914250135421753, 0.9136798977851868, -3.8976635932922363, -2.0292701721191406, -1.842231035232544, -2.610778331756592, 2.5141425132751465, -1.761977195739746, 1.7404600381851196, -8.848752975463867, -5.644103527069092, -3.679253339767456, 0.8234276175498962, -0.9516597986221313, -0.5828317403793335, 1.2737503051757812, -3.8777084350585938, 2.669919490814209, -4.1650166511535645, 2.3168599605560303, 3.099181652069092, -1.6190110445022583, -4.98060941696167, -0.5421348214149475, -7.714372634887695, -5.081423282623291, 1.9850199222564697, -0.8087296485900879, -3.3872671127319336, -2.980211019515991, -1.2612394094467163, -5.231776714324951, -1.1471772193908691, -0.5937368273735046, -2.4824280738830566, -5.408419609069824, -7.91653299331665, -0.4645233750343323, -1.1552718877792358, -0.4179997444152832, -0.6929605603218079, -2.027806043624878, -3.974682092666626, -5.392381191253662, -4.466598033905029, -3.240255355834961, -5.657034873962402, 0.16046248376369476, -0.12429658323526382, 1.2504942417144775, -1.1238080263137817, -2.6251306533813477, -0.6732323169708252, -1.558773398399353, -0.33028391003608704, -3.1726880073547363, -4.366833686828613, -2.093670129776001, -5.3711113929748535, -0.6123660802841187, 1.1035513877868652, -0.7748004198074341, -3.1075034141540527, -1.0164976119995117, -5.395028114318848, -0.6528981924057007, 0.3125183880329132, -1.4988136291503906, 1.646055817604065, -2.088810682296753, -6.846345901489258, -5.109405517578125, -1.2595252990722656, -1.2514680624008179, -1.3320037126541138, 1.1001068353652954, -1.0953887701034546, -2.907845973968506, -0.8417915105819702, -0.5854113101959229, -0.6489495635032654, -2.427391529083252, -0.5313088297843933, -0.7208320498466492, 0.8016018867492676, 1.0251641273498535, -2.9831597805023193, -4.509387016296387, -0.6843605637550354, -2.19754958152771, -5.548168659210205, 0.8586586713790894, -2.2243642807006836, -0.07158086448907852, 4.018250942230225, -0.7129886746406555, -1.9302146434783936, -1.1506508588790894, -5.521100044250488, -1.937376618385315, -2.8662688732147217, -1.6104379892349243, -5.726207256317139, -1.977686882019043, -0.7568149566650391, -6.316845893859863, -3.68035888671875, -1.6397846937179565, -4.497854232788086, -0.3983043134212494, 0.46915918588638306, -3.7764012813568115, -5.36472749710083, -1.7140735387802124, -3.94818115234375, -4.093946933746338, -0.9790458679199219, 0.00029802179778926075, 6.426813125610352, 1.2088385820388794, -4.278616428375244, -2.2354958057403564, -1.4396557807922363, -0.4497317671775818, 0.4863741397857666, -3.5141518115997314, -0.3134520947933197, -4.97646951675415, 2.590839385986328, 1.8613888025283813, -2.070958375930786, -0.6414903998374939, 0.18730032444000244, 1.0887341499328613, 4.258052349090576, -3.3800668716430664, -3.1285085678100586, 1.5183186531066895, -7.245822906494141, -1.019460916519165, -4.9781012535095215, 2.9752731323242188, -2.0180864334106445, -5.453919887542725, -6.236088275909424, -6.162588119506836, -4.98362398147583, -1.9796514511108398, -1.4056810140609741, -2.3585848808288574, -0.013642038218677044, -4.508732795715332, -3.169292688369751, -3.0666561126708984, -3.421527624130249, -2.3532276153564453, -1.4291812181472778, -8.265231132507324, -1.2493188381195068, -2.606353759765625, -5.808748245239258, 1.0636711120605469, -3.0066311359405518, -2.9893743991851807, -3.7193543910980225, -1.7736951112747192, -1.883446455001831, -2.2648699283599854, 0.8642212748527527, -2.3063857555389404, -7.84710168838501, -3.32696533203125, -5.64229154586792, -2.60528302192688, 3.598360300064087, -5.22620153427124, -2.126671552658081, -4.361355781555176, -3.0411722660064697, -1.2308878898620605, 0.06109987199306488, -2.7101011276245117, -1.6204407215118408, 0.2793930172920227, -2.368961811065674, 3.8008391857147217, 2.6374404430389404, 2.3225595951080322, -1.2882227897644043, 1.2367347478866577, -1.234601378440857, 0.5947901606559753, 4.069944381713867, -3.6015126705169678, -3.762533664703369, 0.7717007398605347, 2.246351718902588, -7.987368106842041, -1.8682599067687988, -4.67086124420166, -2.8297548294067383, -7.317141532897949, -0.7220501899719238, 1.5564504861831665, 2.4207582473754883, -1.2658201456069946, -0.6777290105819702, -5.824297904968262, 2.219686269760132, 1.677865982055664, -3.635587215423584, -6.389676570892334, 0.8020912408828735, -1.7361177206039429, -2.7722814083099365, 1.2519201040267944, -3.8913283348083496, -1.8172979354858398, 1.3850769996643066, -1.401163101196289, 0.6747341752052307, 0.545479416847229, 1.2535942792892456, -0.7016334533691406, -0.6863641142845154, -1.3954054117202759, -5.71262264251709, 4.915378093719482, 3.870577812194824, -5.968655109405518, -1.3520456552505493, -2.8018887042999268, -7.219702243804932, -3.26814341545105, -8.865581512451172, 6.02556848526001, 0.06614404171705246, -1.5633370876312256, -1.2734991312026978, 1.3058291673660278, 0.6659774780273438, -1.2890876531600952, 1.2852061986923218, -7.53849983215332, -9.33144760131836, -2.6095449924468994, -3.7027077674865723, -3.672733783721924, 0.3350190222263336, -3.9193272590637207, -1.1023012399673462, 0.6867563128471375, -1.0231072902679443, -0.9067835807800293, 0.7320749163627625, -2.084770679473877, -5.743011474609375, -3.9618098735809326, 2.3502538204193115, 0.44874265789985657, 1.7804181575775146, 1.3197335004806519, 1.6181886196136475, 1.8429460525512695, 0.7967718839645386, -1.0954155921936035, -5.223276138305664, 2.0012505054473877, -1.6384540796279907, -1.7337393760681152, -5.072771072387695, 4.019883632659912, -3.441208600997925, -5.775191307067871, 0.5979641079902649, -0.05256747826933861, -3.655266284942627, 1.4014562368392944, -5.759852886199951, -3.0958359241485596, -4.769505500793457, 0.4112968444824219, -6.249650001525879, 1.7530555725097656, 0.4753119647502899, -6.25303840637207, 3.5254924297332764, -2.328989267349243, -4.392485618591309, -1.5876281261444092, -0.06800958514213562, -3.9386868476867676, 2.29876708984375, -4.854025363922119, -5.8848161697387695, -3.070946216583252, -1.956011414527893, -0.7474510073661804, -3.417623996734619, 1.5151618719100952, -8.113716125488281, -1.18203866481781, -4.497945308685303, -7.051929473876953, 2.11665678024292, -5.036048889160156, -0.990921139717102, -5.974513530731201, 0.22866122424602509, -6.0758585929870605, 1.1424676179885864, -3.087442398071289, 0.16313789784908295, -2.5759315490722656, -0.7621667385101318, 0.5372371077537537, -1.0577504634857178, -5.371982574462891, -10.08492374420166, -4.9135589599609375, 0.19328755140304565, -2.644160509109497, -3.071596145629883, -4.637571811676025, 5.406352519989014, -0.478708952665329, -3.5495779514312744, -7.116058826446533, 0.40589332580566406, -3.8400566577911377, -0.8950452208518982, -6.441202640533447, -10.926152229309082, -1.3576281070709229, 0.9105145931243896, -2.2159695625305176, -2.570502281188965, -1.941975474357605, -4.196835517883301, -2.3903257846832275, 0.6704534888267517, -3.1005921363830566, -1.7582892179489136, -0.9247517585754395, -2.695591688156128, -6.809054374694824, 0.5147160291671753, -1.6173609495162964, -2.4065299034118652, -2.247587203979492, -0.503130316734314, -2.3780741691589355, -3.255816698074341, -3.963987112045288, 2.150386333465576, -0.9333065748214722, -4.803703784942627, -1.3913954496383667, 1.6655373573303223, 6.960179328918457, -5.698686599731445, -7.977048873901367, 0.9807363748550415, -1.3686017990112305, -6.023014068603516, 1.3016127347946167, -2.5733144283294678, 1.315368413925171, 0.5987314581871033, 0.11382883787155151, -4.110584735870361, -3.1384880542755127, 6.161542892456055, 1.7283328771591187, -4.865611553192139, -4.443768501281738, -1.9632251262664795, -4.388583660125732, 0.03750063478946686, -2.7605509757995605, -0.5727862119674683, 1.4782906770706177, 0.3242006003856659, -6.363930702209473, -1.9101746082305908, -0.13010859489440918, -0.8922516703605652, -2.583205461502075, -0.6538357734680176, -4.978386402130127, -3.8793582916259766, -4.715310573577881, -5.061905384063721, 1.08785879611969, -3.119875431060791, 2.197915554046631, -6.659328460693359, -3.2872440814971924, -0.6228771805763245, -5.054793357849121, 4.6327009201049805, -2.125549793243408, 0.75328528881073, -1.7013176679611206, 1.732019305229187, 0.9104007482528687, -2.1041784286499023, -0.7946922779083252, 2.174037218093872, -0.8887652158737183, 4.115453720092773, -0.37063008546829224, -3.219860315322876, 0.9312073588371277, -1.9281809329986572, -5.541512966156006, -2.1025712490081787, -0.7921200394630432, -2.6100077629089355, -0.9966434240341187, -2.2937166690826416, -1.810380458831787, -2.9926486015319824, -6.433532238006592, 3.3596436977386475, 1.8334301710128784, -1.1478036642074585, 0.13521525263786316, -2.913224697113037, -2.73564076423645, -0.13548138737678528, 2.5907998085021973, -4.384252071380615, -2.4670071601867676, -0.23520416021347046, -3.279297351837158, -3.8026282787323, 1.887200951576233, -1.3610491752624512, -1.6791430711746216, -2.7519354820251465, 4.300793170928955, -4.8734917640686035, -1.8522613048553467, -7.383311748504639, 3.8919005393981934, -5.044241428375244, -7.145243167877197, -1.6019887924194336, 2.3383448123931885, 0.21400241553783417, 2.0545732975006104, -0.9127764105796814, -5.95515775680542, -0.41230419278144836, -2.215761184692383, -5.859074592590332, -2.2364187240600586, -1.4933149814605713, -2.1224448680877686, -1.7600263357162476, 1.9237523078918457, 3.9585249423980713, -6.36295223236084, -4.216599464416504, -0.5297714471817017, -4.261625289916992, -1.9895621538162231, -2.226832628250122, 1.6938755512237549, -1.9112608432769775, -3.764868974685669, -4.397538185119629, -5.872534275054932, -3.763423442840576, 2.6267011165618896, -2.810238838195801, -2.169909954071045, 1.3268051147460938, -4.8943095207214355, 0.13085009157657623, -2.2981462478637695, -2.96628475189209, 1.2636611461639404, -1.9606289863586426, -7.624131202697754, 6.11342716217041, 1.637393832206726, -1.0160212516784668, -7.940097808837891, -2.981393814086914, 0.8830887079238892, -3.4283792972564697, -5.104644775390625, 0.7757311463356018, -0.39970311522483826, -4.775166988372803, 2.0884580612182617, 1.08719801902771, 1.3341693878173828, -9.012675285339355, 2.083491802215576, -5.865126132965088, -1.1889591217041016, -3.0887386798858643, -0.9942315220832825, -0.5284069180488586, -1.0525349378585815, 1.244728446006775, -2.4495887756347656, -0.9892718195915222, -1.6242293119430542, -6.7879157066345215, -8.40389347076416, 1.5399636030197144, 0.36406686902046204, -1.8547344207763672, -3.2567827701568604, -2.21718168258667, -0.9367918372154236, -5.896705627441406, -5.168668746948242, 0.545005202293396, -4.0823493003845215, -1.6033294200897217, -2.359663486480713, -1.4078736305236816, 1.7452210187911987, -4.029671669006348, -3.415187358856201, -4.195060729980469, -3.6820993423461914, -7.535355567932129, -0.6622709631919861, -5.957634449005127, 1.0925531387329102, -7.5496625900268555, -6.960525989532471, -1.461761236190796, -3.51412296295166, -1.1333765983581543], "dtype": "float32"}], "e": {"__tup__": [-1.948801040649414, 8.845864295959473]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(mean_var, json.loads(_TESTS), 'cuda', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")